# AxisForge · características visuales congeladas (OpenCLIP, DINOv2, DINOv3)

Ejecuta todo (*Entorno de ejecución → Ejecutar todo*). Procesa las 352 vistas de desarrollo ya selladas
(22 objetos × 16 vistas) y deja, por cada modelo, un ZIP de características que el repositorio importa y verifica.
No hace falta clonar el repositorio: el código y las imágenes van dentro del notebook. Con GPU va más rápido;
en CPU también funciona (DINOv2-small tarda menos de un minuto).

**DINOv2** (`facebook/dinov2-small`, Apache-2.0) no necesita cuenta.

**DINOv3** (`facebook/dinov3-vits16-pretrain-lvd1689m`) está restringido por Meta. Una vez por cuenta:

1. Entra en https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m con tu cuenta, lee la licencia y
   solicita acceso. Meta lo aprueba a mano; hasta entonces la descarga devuelve 401/403.
2. Crea un token de lectura en https://huggingface.co/settings/tokens.
3. En Colab, abre el panel de la llave (*Secrets*), crea `HF_TOKEN` con ese token y activa *Notebook access*.

Después añade `'dinov3'` a `MODELS` en la celda de configuración. El notebook no acepta licencias en tu nombre
ni imprime el token. La redistribución de las características debe respetar las licencias del modelo y de los
datos (vistas derivadas de Google Scanned Objects, adaptación MuJoCo de Kevin Zakka, CC BY 4.0).


In [ ]:
# install: dependencias, sin reinstalar el PyTorch que ya trae Colab
import importlib.metadata as metadata, subprocess, sys
wanted = ['open_clip_torch==4.2.2', 'transformers==4.57.3', 'safetensors>=0.8,<0.9',
          'huggingface_hub>=0.36,<1.0', 'pillow', 'numpy']
try:
    print('torch', metadata.version('torch'), '· torchvision', metadata.version('torchvision'))
except metadata.PackageNotFoundError:
    wanted += ['torch==2.9.1', 'torchvision==0.24.1']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *wanted], check=True)


In [ ]:
import base64, gzip, json, os, pathlib
os.chdir('/content')
payload = {'research_grasping/__init__.py': 'H4sIAAAAAAACA1NSUgpKLU5NLErOUEjOT0nVUShOLUgsSixJVUgrys9VKMlIVXCsyCx2yy9KT3XyB6opStVTUlLiAgCOdSKYOQAAAA==', 'research_grasping/phase4/__init__.py': 'H4sIAAAAAAACAwMAAAAAAAAAAAA=', 'research_grasping/phase4/jobs.py': 'H4sIAAAAAAACA41WW2/bNhR+96/g8iKpdRRfYsd254duTdpgXRusxR5qGAIlHtlsJFEjpSROkP++c0jJkZN2mIEgInmu37keHR1daSi5BibgBjJV5lBUx6rIdn1WanUDBS8SOI5VXQhWKl3xOAMmc74B9l3FJjw6OurJnF7YlpttJuP2KFX79d2oov1WppdqlbOSV0TMmusrPLYkGtqvCvIyldn+fC8PjkWdlzvGDStKJ/Tq8mMr8JJM7PX+ev9b9PHz73+wpVXhRxEJiKIg1GBUdgN+EJL7RWVWozU7YR7eA9fJ9qREf+D0hNeVynklVXGiN3F0I+HWhJlKrkNyy9triD68/fIB1Xiz4UBwEGKQjkUaD8fjiRjDkIuzeALJRCQiOeUjPjobj9M0nsV8MI+BxyIeJjGSwpnX+/Ptp8uL8y9f9yKnyWQ0H45mM47s4+kknp2N+Wkyn43j8SCep8N5Go9ADIbDAR9MzibD4WgyPI1n02ki0ul04PV6PQEpM1sexbsKjC94xYNFj+FPQ1Xroo1eiDSjydQRhFu4E3IDpvKDjgiC0KcANhK2aOMz9sA+3Mpq63C31KEqofA9HXsBRS113PRLlWaYCwWTFWg/43ks+IKlGCQu/OFgdMpeMfoX9FnsecETo1Uf1iWaC34cHDj0A+sTXqhCJjzzb3hWwyECFM9QYE4Z99pnBjMpuoadWX7V9kylwiulzdL3+l6feQsPTeJZpm4jrJTlBc8MBCEUiRKw11q6EvO1UlWfYdWAvrEZZfBUV2WNt6/o4TsklVl+UsXesp9ytAlNFGiC/e7S7e8stQPGqFongJzERLme80KmiE+TykQjOyF29AH7ZckOUvIJfs2lAfY3oXWutdK+R3ljoGKtaCYUYIEquqmSLULIMxD7Z89ZtqdeuihkigvT6LdJEFVwh2F01BZvFLJkD3rlOdgiKbz1gmmbSppSqZXZUhhvTd4hhykzWdGpYL5XoQ8FhRJjLoUFzwseHV6QIZ9VRJkAwm80BySokcqkYRQxBhj6lq55C1pIyf+9NDQwwzpAlPz2LrAgu9vmhsgcW4eM/bq0F60d/xWJ1r68RlxjYHUh/6Ekft7o2eU704TBNtG2AUYbzU0pi01oO+E4vJEGwYlkUcFGy2rH9v0a40Mdsc9uQMt0h03S1DyzIhsmeqbEa0mbBqHVLWa0HScGn1frPntw2FMYlRQUo9b7J2cbRDWEaZ1lNrF87a3eHn/jx/eD43m0fo0RRfZnneIlSHVheApNLBGJBgj6lWhQt6KwYHyy6DXzwqK871AeeO2XVnO/6/kTqe2IRWkTnChd6yhlcp1B0z2oNVIVHZpOdVlp25ZX+7qNXLf11i9rlNKnwwBme0jcZNAKTV2vPBz1CRgDYk90qP1nGbZHp7MssFwaG5IOQpZ/EyOizh789tbP/cPLUFS7Esg+xKjGTJuRH/SAdrkHfzjts3nzNw7+j51wV7rKG04ZjmtmZzjjFYq4m0+fm6luqbF02srCxbNpG4tn0DXNBN87cLQwLjrzMqAeQ5rxdrV+PFBK2S77WH88B0p5wNUGNM009P0HPmJkgedop1ThbzTQLz/7wRu39IRUxVxrvvOtvCA0HPccx9InVRidpXf16f0zz+lXkAFLlnquJk8e0MnHkwe5GIzEY1gWG+8NizNFkXQCww1UdlqSfse0IiFrpCDCl+FRt6sGh3XIS9wIhP9gLxzWCINH2wJ+khzCvYulW19IcvD4InCmFYjfzcjDyNtwmmQLOY+wVKkqUdgQJTfDKnpeT4vDWrKxdSNkYdU87mWvPNyC7ewhRPb2Pe0ZRNSMLTeKm3UzzK+F1H6zezb7BdxJNEJd22PTkjHzaA1G8e02jKyGvn0UsDwQSjs7pPJu6YVo1jH2QFOn7oyrs4cRUiZMMmXAT4WTX+nd4rA7NUt2+E2WF5S4pAoRuEVpicpRgSEEl3uyy6vo3fnFx7dfz9/Z3nV/mK734S0OC6BeRFi5VaPPfohQtxxc8G2u0V5oMytEQbnxg0VX6hPhkxB0E3eujCet+d01KJUFlvDuYJwgAyVdaCNgLBNqwdu6yGRx7S66yyJZ3fsXtCtHHj8NAAA=', 'research_grasping/phase4/bundle_io.py': 'H4sIAAAAAAACA61abXPbNhL+rl+B+j6QzOloyXF8jV13Juk5jadJmkk6nbuqGg5FQjJikuARlCLZ5/9+uwuAJCRKcTLVF4l4Wezrs4uljo6OXsplkfKUxVVyI1acreJMpHEtZMHiAoZrmYuEiULVcZbpcTlnKV/xTJY5L2o253G9rLgKj46OBiIvZVUzIe2vT0oW9rdUg3klc1bG9U0mZswMv4fHIXsPNN5LJdb4aHeom2Utsuapjmv7u+Z5ORcZt893wnkslnm5YbFiRanPBAY5yhgtqliVoliE5U2s+Gn4Sc6U5eTti3fXr64+/ha9fvHx9ZB9+Pll9ObXn35pf5mJJC5kIZI4GwKHcTTb1NyI9v76jSV2nccLPhi8ffHv6MWHn15f/37FLtmz8Ql7wsajk1PzRfNvr96+vPoA02enPbOvrt9cfYTJ09HzswFyAb9RSX4UochRFIQgncxW3A/CMq7AKGpyMmXHzLNSH5OwT4/BsvBdH+dxIeZc1WEmk9sQbeQNBoOUz5mK5zzKeT7jlfLvgvMBg48o5lLBqXch/sqEqv1AT8xZxguf5gP2I2vZlRVTy9xfh8SiEneczWFsDbSYs9zoRh+EnyoWirPf42zJr6pKVr5nfZOvE85ThbaUyyrhLBO5qJWneSninCOTIJ/hDg/Es5oz20NwMazFUeIQny9YiZrt+qGP40GzC8T1C1nr3UDc+/NPD4nb5zIUKopnYItlDcbQQ7GKSqQHz99dtlvDkLaWaLFa6aHuSHOo/cAK2mzOoy3EP5yZikofhyESfoyuP7559wuZJeTrmldFnEVxXVfsxx/Z+Czoo92oQhtLG8c4Jkz7W/MIDqgKGk5kXoJRFE31Uvd31xGJLarHPfSAlWejURCcO3R3vWRZoO82QKZ92Asck6swTtOOUSsO0GX0aQIgk3EaASj4iFLm0M+ivrEIE/4hylfwrecRYu5ctyIfdKOo60Ae0NYRZ/SnN533SJQLpQCpGOzoyAFPcAJSCJFXoA/hH6d+SzhoF4NSxKIAfL9kqUhqH9YEF+yTwAE7F5aypN2RSL0heycL1+VxNXhug3R+A3++JRGQ8yEDC4g+z8BMZGEmgr0nz848igAHZL9kVBQWoOsGyVtqDPSSx3Vy01EK4pirFQRKUkxUQwj4gSNSK0u7TD8HPTz28AWwmoFW6dgediBXys+k9ftq4snZJ57UqN0p0Jp4qsxE7U0JnzDwiIxdpmACOOwsgwW+VwMPBVjHaxO0Fzz0y2Sz1a5cTh7rkwsWMJStT6hqMYt29eweZnQ98VaCfwZRWgbTBpsvAMhLkJSnkY2X+9Z5W5lQPaATlB9mO/pxnUaSM8Oko2hnCWgHV2EaSAk3jX3IW2GGdEMUjM77VCMKUj3ThxDY0mIX7+AAwhik6kyQQiyj29oxTE5WeookIPlXyDStniKPlHqruFhwH2A8OMQmbgImUwcDm6w41PNAnEOpxKsY8hUdswWznVw59wRWNOr4HmR7OL4X56OT9CEsi4W3swVkQWoTD0HSm3YTH313kY802fiuATOCaDKLpmMAZLrLXT9ukHQ9Pmw/hOhUoYWyxApGhi/x/OtfXQ4I4Enu/pNBUJoNQatwFjLsvX/3s0cZlSZymXIahkDpDFNmg2H/+dmQPT87aEra0SMEfkqx5hn6VYGVRlxV8can9UGfUVo90/JEFrVYLOXSbNTEgrCWXdCwcT8Bu08nYroPN0jn+8DDftzQ30rHhkvyDh2o2kOACXffeW/+t2soRZrs6zmJHiZMmjcwyqOivPMxX+mIiBK4D9WXYzBKLW95ES24zHldbS4pLR4oBzoehOSCbygNUO57b5HJGZRruCJNIf2rsCg3hPwaGfTzA/vhso0fW5QqHH0sCXgu0Urb6x6+lJStX757/8eWmrer7u6FwaWqdWiCD9aQujSx3UhbwQl48yQvh5ujiTadcsDZReLrrcGgF4rsdgi38ZCNINggEEo+ZNGQpfWm5P2UKSiiG/jJq2gcjcwhQ6hE1nYY4/hyPIJPj69nW4effOvhJ99yuOL9YaKWJV5QORrwP5Y/rxcviL8QCrA26eF9jwTAC9xTSqbFxsd1fhEwoQy209qC/cBG5BB4AdNyB+eHaneQmWnhehgizMRLW+1rxkTN8/4rh3tmP3YTuSeg/H3IvnMROu+9nCLPtZQsi6temG4pubedPmp1tSySuKaGDNA11ChY0Eeg4toBmqGuZqJSJLcZv3wVg+EpmuJW7NXQFGDxZBccvOkQxy02dIs2cN+wSEWOCHyCFl2FpNDJaGpzs0ZMC0IjsLhdM54iGn0/fm52apvdioKuE97ca5AL781zUQgsQoIQxPGDx2KQloZ1pHEgFbGOuKFc2/I7pAsLzna4MlWJJ5Zd1nQg8v8u4b4DG4Z6jGqwll7waIYpS17/a4vPHSSmVkDsEi21AXeW7tSRZWO0p7oHcdhoPlqtdKyGvSa6oHenThqDmtbGoyxa9lr0sJJIxH6jGhHd7GxxB5P07jmLiq4I7hZ9U8WpiKTrD1xyCUXtzyLhPi4fUhUeWCzEoaCJD8TCrT3rISIWrV+j/kZtI0zvhSf8geZ5on+NyVCtRR5b9Wq1kbidEswUPLwS8000WxYpVCv6a4gFUdRpdOjeQtMCaWb3lz2a0Lf0QcBr+PorOyG0pxs5+NzfDemQd+/+eoKsr5IbnsdRkwJR62PdDGvWmL4IzdE11AxMt9b9dT0PrdJjKmJ3C2l0nlvdE8CrRaabNuZ3xVeCRIERkC25LSX4nmUFC75KJlwpWXXG3LjwtgLVBkEj6W1wyELEB4NjVrxA9z9nHvs7u235hzvHXCwojTe602PAy/3DlqB8o0VtulN/idC7ImoOtHx8QzbrSAwj/fcz2KYADPA1yI7IsKm/PdO2z/SpwdZxHtQA5HJ9Z8KcYXZZ6dcx1B7b5yhWf9+klxWeay01AVrTwWF4pA1DpuqqgUcaIgHPTi1AJk2mHY1Pnp4+O/vn98/jWQIo5RHXCTU9aOOha3GrcdJBR/TmIgiXIbnVf5Nu/03utpceLuAWy4u2XYWAp7tUHVBx+1SV/KzfMqCmDneqYKnD0kXbYLGv0nSLBS5jd96ePhYxCOdaSZ1OFp1ghdTMTTwU4gudmK/J0BqkTDNsK3cic00TzKivvelfsFlGaabbaBnsbVTg4o5Utg/Uw5rR3t6AMC7dXvuR9M4935hwCxSnbiyj9r/T/mEtQB6vPcXoWO3DjLzMeE0m6PQsjEKJgtu0II5MFjcvYnvSOFwmIeuIgkCheW+n0+Pe3K/P6Wy0rxa7tOybvs5YyNcAe0q/c+qOCxWpTZ6J4rZb8P2NfeBLUIQsMrgq3eCbvBhuk8SW4KnzehmrpSRbYtlHS1EhqOiwa4B9R9r6c2ueXo99uUAnJlpTYOVjC8uYAQ0wlKw2e3DGwQb2P3bfjboDENHiiJwpXq2IWvsO0wKMZqZwJKvw8uN7T3ryGS53jdH3EkNPaq5c4XcLbksRJUJyllsKbJqsOFpwxaNaOv7TeQPqUqX+CGrX0tZ2Otzb2zETJF/HKAiSVpOdxiG9hex6+rFTggbOyxLc2KZpWvZF/6GQ0EVq588SqeTagwiNdrX76OThgOKWIB1XC3Zf+rjA+QiM75cEIiGRVbUsu3DvQNQWlpj/IoT5LdrV/DHh8rcKSwQ6IpK39KjJ4T86LPzYf3fgXvztlxWfi/WlF2oF/gPKFyB6uXuacTFwiVbSr7i4fJVJ6E6OcXmpmXcMcaG92tFBj9TbtMLPFVya3ZchXfO6u3xzruvJXRLbXtzulwqoY54iIm760H+fgKcs6+gRr92wtAH/c/P/nLDK64prOj2Z6/8ZLTvIZiQAAA==', 'research_grasping/phase4/orchestrator.py': 'H4sIAAAAAAACA7VabXPcthH+rl+BuB9INheqdtpM5xR9sGPZ0dSRNbbTaau54VAkeMeIR7IEKenk6r/32QVAgkeeo0wmmvEcASyAxWJfnl342bNnP2xkclNXednKVKTyVhZVvZVlK+R928RJm1fliSgrkVRlljfbuK2aHcZk0tFQ+OzZs6N8W1dNK/Kq/ypVLZO2b/JPkV+HW9nGadzGduQXVZX2u1JHWVNtRR23G9CaWeISTUtSF3GbVc3Wtlu5rbO8kH073/bfD/loqOy29U7ESpS13uby/J3d4nwbr6XubaSScZNsonUTqzov12G9iZX8a/hLda0sfRKXVZkncbEQahNH17tWKv1JOy7ETy8vzt+cffwU/fjy44+/su51V6aFjPLKLl5UcRpht4W4jYscwpJRWT+gJZs820Wa/ujoKJWZwGVs88QniS0EyTVYHgn8UYc4ZdHxYHDCXWEdN7jZcHuT5o2vG+r0U9OBZ3mfqzaqbrgZ8CpZuhDttsZCVs6YqejbrxuZ5fenXqg5+MbD9nlz6myil2ibneaI/u5ycFWpMEurWpY+Le/dXXsB3Uq2FFl41+St9PkcJ2hmRac2Pj5pjtqViY8+cFFWfhD0q2KwkVCMRPpgdsEHNfznZVwUDgN5RtTMJB9X0YxgSZ1dWeTlDbeNbKOmK6PBBnxcSaQFfVc1NwuxrVIJBfiz+YoaeZsrEC7IiPJEnnpJ3UEu13GLa1f5gzz9u7ke2MxPNEdsO9WKayliUdP0qlPFjhVAQjbQmwdZCthL3cpGS69uqltZxmUi2fDMocqq3WNCVI2zsfhePF+KJs6VFP+Mi06eNU3V+F5PHZepqCuVt/mt1PMEz2vkf7u8kamnBQoRQBmsgvYSwQ2RSKzC0bfpeqqm0bEwnc8QDmfksaT3T6CgsXAtW9+7k/l606oIVvfib995/TqJVKpqIhjXZkQ/DNkZJKPefH2ivBqovFUoywT8WFWDdRaS/GJM2mDPahwdbUGa6be7Wvp8jCAYLEDAZ+IqYMFKz506xNAM+h7TmvPI+0TWc/4zvIyTGziti6p9U8Ej8IVO9/G6Mr6N8yK+LqSnpUlefI2hz55RLCuPZe/A/PFZA5gprSsVbKGt5iZE/BNFRDriwdvjCePsiJ3xsg4j24wiEChstY0diufoJF3LUzTwcWVbKwxovechfYV9x2rRG77+88Y2ghn7lusNyjaccuibLDhRquWeCvYMDkptGJ2sNZgrKIYGVtD+BL36Y0H7ysSewctgkO23L7xH1x0MTM8Z/jZXCjHItS3i1mgdOb48hY4M1tFHPF8rUAADVy3FJhNq2AEc4/KoM6Sw7ll2BjrrdYPBJfMg5tOMkDyL8h36RsLVtHDCrsO3S155mlFvJb46NUzPHTaHHW+xXA4joDjcbaUxg65hBbfGVig5Zeyz3WVptiAt4dmkGvyx0MfuFN3G5dnF6/OLtx6TkR0BVqH/8yM6Mpji0NrETXoHz0gKYXBNaD/8QN+mOahZfkUm/eHniwva4MRG/0FeiwGZ6N7gUBQ24Cj8T16/IQPuXTnF4oflSDfBj6iufwGo08aHb7g8cLPcU2HEYlYbEFgqvh4E8K51dCSTOA0uwkMj8z5j0mMIiONNlqN4jRWxsZHDINGZzfkOS5yVNHdCf4WVVrNzjMWAxzBX2p/1sYFbGAlIxXhxXIY29QMcsAFNdDCpmqarCTnC5FLWwgKW6Ymv6YTB7Eou/CMetDVoe4SjbasbWUZrWSEmNLtTw9y411vBTqGkbV52crIJhNTQvRBqDmvZZFGCWIKQQJhrrTByhbur7YddFM2LqpwulzXxVjIxXHqs4qaJdz5D65DxXl6Fr4j58/f+A5/Fp4ALpQOXQRCCTbh8ROoPb195QcBqd0t3z+p0m8s7qNz0BgFbsOVfcMyuIZSBxuA8J9R3G1woz/leFIRBmedg/ipHZjO5Y77CHrQYsKDXu8IGS9rka8vVav6KTXz/AKnjDlhXyADRfXhjqKtH9lRlYiu3SMY8Vl82ksbH1AB+9I4ukZS4l8ppDwEPrjxIcBvf+4i6tuP4WLwIDk77opgMdOLkjIDAlwkzTRQmXRqTMfbgBSHDHUIK0u4iNib/MGNGuue8vUFIdazUYQEcshTDXG8BuWKZkxkwdu4HKBCxYkwt8VBoKhViIkmZJ/RLefPncozw0EZku2Fcw+RS39Ksi+o6LiK5vZZpisiv2DPUE7qaTWefjDT5lO3lS+tN+IV6EtYdzB5rwTWo+FY+ROSasZqSqQ+6hZiseApSSCeBByzxz1+rgJO7EX/7RDURka9A4OHBuInLtfSff4e0jMA59QHufPsimPJ7XXB+A3YIz9/SPRHLIz9MNEEfemGHC54WzDnXuQjEiGIEoLVD50UACmQR15BJpCSOlRKimPHO4hvtuxltjy5/2SvI44SjJ8OFA8Djh/c/Xb47+3T22juxwwbQ8Kke3YzljH84tdxzZzMLX7788On85bt3/46GLRyU54iQMZrw3rw8fwea/TUtN1deEauW1+Z8jH1iFJVwzVF09FvEceJ4zCfSS0Abg1dMIeEPrSN8lE0O/USuzsWTRkE3S0HFPAYcwBomIQFnhYwVAnTeKoGATNi3XH+jdlTRQVKf3AwlBe2ws6RsC+74nek9Q07fwj9d5mlC2tILNDbw4q9NIYh6B22ZxBbmKcyIymeebUFoYYbevf/hH9HZv8T/3PbFq0GzjYa+otkQwPn7w2FXO2w3OoNROP6NbJzqqKjuSiXaDYICHxEiAfKlwBwIrv1h3cMHMgrzxGrTWEVOD2iMzRMdlRk+B0lM6mNPF/DPF7ZMJu9JV0xZcsI2HGQN9Lz87eWjmcSQh47dFDOYZoiD3+hdzFcj58XQ3noWzuNWQ0GBaEcVhkE40+hN59HJJbknWtdNNW0yXd25ANoFz09IrA4lVL+S55wIJy0/HhKu49l0yyRAnHHPZ0A6NZxJgX4t1XFQzCiMDtm9yWdG6T0Lag5jOSz8Zng1A6zMhRxYcbh5XKEFSp+de1jS9VDyXxd5Sy26Jt2gyhRJDr1ZL/7jQfY0y4KAPZnOggBDsz+wetSHSVxV0DqtjaFM5b2GHHtFNbemtl9SowKjkm20jcs8k2oohY3eNUa1t2RUd5tW2vrxvms1X20D4bR3tZgrtSVX087VbMGt393pXM2AJ/pCty347FWZRiUbM9Z3rJyyUEIFU2PLS1IeDY20L7RuznjGE9P9BzzNaJdMTzOqy3TXQ16bl5rRrvqJJSkqhQw2fWrNiF9bvDssaLE8haSe6vwy+vjp/Yez13M1pQcNAihj9VhJtT93IRV376F08pgQKHlMkuvSrmOxBTqvtN3hPpyG44XcVzR9BCcO7b8lmUv6Xa9JTpDnIx0d/Ul82lAlck2eqQFEe/3yrTZfRQWKZGPhGZyX2CBbEgAFXVyYV1fKhxnhrTVNiAUv4ZuooFtCzZXApTJBScPENOUoyY1MT0SBPYkwZlKRVlK7WSg5gGF4RKzAXxyZ8rYkBaFoDR8ECKC7DUyJ+rCCsRHtwlKOpM1kk7k9rfP0HF1XTDye3lNCR1IdStayNIJy+QMSA/wiJYokZXE9wfzMft16s1M8y4QqO2t+vX4a3RxJNdGC2DtHT8b1ZiMsZ0o/HncpRxG/p6ShRwOz+uiZyqSIifNbaWvxfTLwQZLTEV2puppQGVQLfpzuW9GTfIt4S2UjRYiuhVYkGyTHpA0AcRA1v/orhImSOOuzAfsIOBdHFuxYzU9hfwcvvzDxcGEfMRbuO8fClPUWzoPGwtPvnvjQtXp8uP/1YHjmwNGsBHTdRbM5BwS6kk6YkEDgQfrXD/cRAPYtC4TUHkXq0avx5iuuqZkRPvqKSmtjoi9hxlcEn89eL8f/nQJxqa2SqmCgQ16CsAbdBlAHQhZMf+8lFvz5lo29K2EM+3yGT4LBjnJ6LolBLROakffdW7IAvak8+h5lcUlB4cWDI6puX9iPb/VDK79JuZMdJHBoE6Myzi6cCXtUAtSrcnpvyR3FWgUWPtLTlrOkS0Ov4oe2NuVmFuV37sCgqFpY9vHNpTEKvLI8UNh2x41ej8aDLymNa9Du/9CZe8UyoUYPHf0fYAlnReAjAAA=', 'research_grasping/phase4/frozen_models.py': 'H4sIAAAAAAACA8UZ227bOPY9X6EEWFDqMMptZrFwRw8z2XZa7ExbzBazD4ZBMBJtcSNTGpGykxb99z3nUHc7aXdf1kAciTw8PPebz87OXj1UhU61C4oylcV5aYrHYF2Xn5QJdto2sgiUSctM1fZlYMpAb1vwrNybopSZjc/Ozk5guaxd8G9bmhM4vQ0q6fJC3wXtxgd47YBMs60eA2kDU3nYWlkl6zQXm1raSptNXOXSqpsYCNClEWUFJBS66pDdvnl1+48P79+++yj++ean6x/++hSaWlW4aJx0gMfGmfYcIBKkXRDPApd5YHMJqMRaF8rjiy0woUEA7Qm9lRsl5HqtjTo5OcnUOhC0ZkP/L1qcBPDRaxAUgFttrJMmVe02L7R1UVDWfpvWFkEttVXBH7Jo1Ku6LuuQmdKobeUePUiAp4C3Pxtdq4xFdMUakOhtoE2HhlafupybKjaZrGv5SNfrbZy5x0qdJrDRaOP+1q6aTG9Pk5v2DQRSqeX1ClaOkakeKpU6lQW///Jz8OZftwFhagkEMgplws8dmgOKv0SnydUxtFv9ADg960COMmgCNmiMbSpUA8mglb7NccE6sVHlVrn6MaTLeLfO07qsWq2QSl0tjQVStmDOsddnv2Q7PW+UE2A2+pMSZeOqxnktC1whVDnfJ144i+vVy6DOeb1PnjsVgpw/qbq0YQhn+U3ESf69+KOBYuBLNoUTrhT2z0bWKnktC6u4Nogyk04KJFa6hKW5NEYVVhTSulbuac7TfYJsd2qo8x/THDVa739M98ck3rt0JTNwhQ0Z0FjaiMmVFS/U2iVhnZ+neXRxcc3Den+e7vGRQGrlmtoEn1lZ6402xDlbLIHjFWdeNFm3SDJb8YBVdZkqa4cd4oB2wBCKQT1sMfY/EmQEFCAe+N+RF0WxK9FjwghQpKVx4P3irmxMZgH5efwDx780p+89fK++gDWlIEIbvKaw582F7Etoo50QoVXFmgdbiIIFD1qKy5oH99pktLBTBn2Nw7GdTlU0+CMejelkQt9AXtgCxWonizCKW9+2GLcyEZK+o5f+ZH9Z0j+1O3h3gl8DZEtFMjy2e/7CxP8bh4qBuhgEDdyC9sFUpHkMqyld5MAV+u/oTCVrCX4HzhRG0VFnRjCyJ59S0HU76fq00sp2EkCJOO+LroSA3i+SD394+2u3+xZP9buzYPz/4BOyitmoeuB2TgNpLWFdRmMDw2QsTlU2mSo+HgLUBBYwhkuMIuEuioUwQKAQRP2OqEdUq+A0CZbsd/I9xtktOIOqbyE6wMtv8vFO3ZZmp2r3G9DfLX0sP0LQLWt4f4eRpsCzKxQX4VxermJEh6ivr7/HdddUBSiSdq/8LkT3EHY5/EWToxopqMqCMnK8QxmeJuxOpw38sQmoNJh9NRQK2pINfaybsWt1n0N1jJJF8B5Effvr2w/gpqoVKmq/yxgjHeHnISGLiyF3pvfhcqqKkOwtRiukZAqWFkUHmW2FISgc+d30BkweHpAk2aWRuW7z0+RosO7Z6RgI0qauQa9QtnWGHPjMEVDOmDHYHUuOJM8ZWag83mtximavXe6dEzS6VkBAqgR6QXhMQcpCSktGDuUryxhEt5d1JsgotirTEggKHyDZZSA3m1xx01ngFCZBU+CTJYGVq0+W0QEFaWFbxRoTrxuTovUBAT360NO4ZD7JrJWEXKYsW8VrKBQxm0AtkpxfHaKGGjHNgaIpgglpbIUyrWChcSq85Nf8ho8wQQpXU6Ghxc3CwDROAMWUEOK+Juk+PvfK4ltsDH2KsGRlW7r0xelQPuIyMJ+6/9rzetqfcjYKio5uiJLkM8uV3uQO4s5eZy5nXyA+Z0QO1FbSuTpEjJwBtSkFMpFSJPM6P6QOdxPCvuxQr7h/9ResDrXZuwdyHI4KmYSKik66UFyMCxq/ScVmXx8c2Tq4bWZJ04InmdQ73c2+pB2KnK/gnFZAyVAAARawjf7xCh9n8lDFXEF9yFDZRrX6+RbdEODIpsjoEEA8Z11ef1PYkSbnG1/X6TeEPI9qyufKy3wmm7nP/i/+QB11gxlYrzVAbGWFfevMTe4wwpC7t+QmyyOJ6DAPcV+QC0fp3Casckcxf75f7OYpi7DdcyolCCrWTm2h/PnyzWkg8G3QKOyHL14QrijGjkXkOssUeBfYzzSGVfk4V4AJr/UmpjgrDuLdYb9b5ZgYoq6+y/5S5cRNhrz4hvBYE2SgFtGZD+cUYOeJM+cb31mBtVxcwC2t4+DzyTTdbcA3oVhMgI6w848DjjgzzVZ0wNDw3YOeGL+MDoKkF2Qbx6+gH7/6rr/ju03+YnO0UmhM36ET7qCQj+VBPfD1zOhvXy745fPJsEuEPfyIysUKqmyiP8SxQFuhc5IpHyOiGGiThycLZvI8b5LLtkWkIhKS7LjY9iWpx9YK7vx6sYKy1O90HricN6CrWSA6lOvgxpQxt9pukZppsY822fqFXWMXqUKQdBTLAjq+LsvOAFoBtkBPTIY8bKC2d4qadTu615toi6Yzl8XN6mUwsLupNXCKW2yVLOnEILi+g98U5Z0sxOiWBZAfp1UD/SrN78AImHfJMVB393Flv/DKnqMh6+xDMlt0T5wNjSxbzJrcL+0MiMZ4vt0KybuEb8qBjPS+KsEDeeD9TfgBX/KuNNCo7ymN2Olir9rpcts+M6CbtWMA8Fs/nCSQYfA3kHC8y2sbV9wRuDUYfoIj0nAge9o7DrPJsAIjPpiAokmFU54o6U6Xjhw8amcDFUEubX7Mxv0wRPBhQNHzBEkZymefBwoBVIxmbCH7Q388//n85vocepX0fqMKkCm0ZtSWqyyxWEpAgBlPLHjHBZX41AWejErlbuzx2bfhQmdswXpqLg4vvCgkaO77y8utUDdXWG9OpMQWB1Lig9eDJbbVDr2C/fZXCUjkaBNsMchCdItC+NRJVdVgJpiUQoYT6N01UEIPN+zIIGQ0uZzOQ8Y7HfRPjSupRvhw0D88b2gYlqYGfnz0K9MWLujg5iPqEUKMZ+HECflU4nzueMenLICH7BE6XGxvt2BaEmLoI3vaVy7Y0PYD9raWwJ8pGLjR/NrFs+H+WVdIZr8nhNWcx6kARrFqYusTy26Jo1kx0hzTDy7h82xB8AVKnHqARB3FUH6EbU86RQjGNuKhsdD0QlGWTIFiBV0DFnkhe01DZowzM8IS9ve3797vbsDPpibHDllJDg2TqlgxRICwjQBelKjFkd9zVzcW5+zbEgIMzg/b8XhPf/fw1QgxUgCbqAZS3URTB9Fhplc2tyJIg7MVHH97L+nu7d4h+43cdwggkx8qjsQQ6D6+0m3QPa2G27z+epQmB5cbm+JolN2Z438AhylBBSMcAAA=', 'research_grasping/phase4/spatial.py': 'H4sIAAAAAAACA31WXW+jOBR9z6/wRBoNTB2aJrMaqSnVPuxK+7LzsA+zD1EUucGAt2B7bJNAf/1ef0CAVI3aBK6vzz0+9wOWy+XfRCJFT4bljGaI1aSgSLKWVhoZAX+vlKNCsUwjkhuqEG1lxU7MBFejCNe5ULVOlsvlgtVSKIN4U8sOEY24XORK1BBBU6JO5bFQREvGi0RRaY3cEMME10lBRU2N6lCAkEr8B7QWi0VGcx/sSPKccRoJxQrGj5q9UWyR4TcLdycl5NGvp9Ear2OMJMmOLxQoUm+JHxcIPsD2H2oaxZFHRS3u0IWZEpWkyldOggC+c7D3AIRMIysKUiiKIiMkrmhuYndyi1niSzoit0OqxOqSjinuUL8tHXHdIWmwrNIrV4fHclQzHgEsdkjxU7pGQjljD4Pdzvhp/YgUYZqin6Rq6J9KCRV9YfxMKpbNc/UldvC6xbpL1eXewt+Xzqa8JlwmRCnSRfs9eK1xpNvVQ3y/WdmQd7I64P0aNoO983agcyeNM6/xQ3I4xCFzZ6bZS0WPkphTGUnBuDlehKoyjDIqTYnRK0bOcLSVcjyRmipi0wb+VGP0Fft6PA700x+CQ66hQk5U6yBsMJ4EN1BUxxfR8EwHoxEVYPITTdfJer29VsBPS45VzEDZceRzQSpPDOWiqsQFmuKlu1b9EHTlNa2JdOW8cJCz6NckvQhjRI0hQGlixDJwsg2nQW+I56HCZsekMbIxO4fpsIhioeycFiuauQ1CZUDYWJloUiQoWiW/Yfv/l/36F77ixIH8EEg30jUWKIgypg1cQ+PZcuKCr0ITSNuW7oTuUEEm9+tESW1haF8azhDv+kyNl4Ip7qt4lkDENLKZQYRnszROlqZy9ks+ewHZkUh4xupP6cYexht0SSTdrw+f0vH9w8F6BHJh03Zs8m6PG9gXfcff4/d6aiiFMBv7uaXor4YpmoXmsp953U6m2IgYjjabbxj+YyvntKyHpd1NdfXJ3my2/bePfY04zsmUDM4rQcyQoRsd31HJjp0bpfqZNAD7JRBwi7exLy9j5wnTcGxmaDR4xgmpqmjsA/enSuiRz37Tz5TD+2B9a7vV4Q7G4ZCFD8Yihy7TIWF2eE+136GixAVY52eeDedegvJzUdrfy+fi8v40PokawOw8HD1ZQ/yQ1VHGpvmeJcwbB7G/4Xfl8V4job0BOuM5DZebw8j+cLVvDx89Ufph5Z3DEdx0n9TcaNxP6buVUanEHwXb/uH9Q5jmjN/S8H7gQ+ztXDjgV3zzIBkCvsGZQ57mIjXnINBj//izcD4UPqfNGbbukCixuIzHSY/cPEETWtxzf9E8p+LSG+G6hOtbbHjhSIF7BFRAGzhvcwcNHOOJ7exsfag5c8dm3+HWN8f1NpyUvOiRz8pqED8PTXJL6Vc6NN7v11cAEME+0mECWcq/xh3Q9kdun9P+xF1v657TcrANhRdWJtXXPg31F8DGRTgjKdupbFFrNfpawHuMm58zVaPOL8MbThAyAIa23ssOy/aw+B9EMlRIDAsAAA==', 'research_grasping/phase3/__init__.py': 'H4sIAAAAAAACA1NSUgrISCxOVfD09FRIrShILcrMTc0rScxRSM7PLcjPA7KL9ZSUlLgAut08LikAAAA=', 'research_grasping/phase3/vision_openclip.py': 'H4sIAAAAAAACA9VX3W/bNhB/919xzR4stbLij8RO3KhAmxZrsCwJuq57MAyDkiibs0SqpKTY+et3pGhLbryi6MOA+cEwj8f7+N3dj/TJycl9kqSMU7jPKb++vXmAL+xz793paAi3b2/u73pn/f7vwDKypECzkMYx40sFgsP1w5/+yclJJ5Eig5wUq5SFqJgLWcADLjv294oovbVb8jLLt0AU8LzTuf744fq3h/ubu8+LPz6+HZ6PIYBufDGe9M9H4XDUH4xH6H9C40uS4DJMwgEZDiY0Qnkyjvqj4WXSP+tfDgejEaHn0Vl4cUm7nU4npgmkgsSLTMQ0daIVjda5YLxwpx3Ajw2mEDJatQUCUVhEKcuNMA90Iu3TRrwKbE6+WhEM2qnFj6xYQe5rE05Xhl1XZ5nU/vQnERKiVcnXwDiwgkonJVkYk2niS0piZ9Afnr3UX64HYbfrTmHll3lMCuqYY7UXlqB4RTcxW1JVOC68COAZjFOQhCkKX0ha0g9SCul0mywAlXoa7IypjBTRqlub/gVuRURSoJs8ZRErTFVfAxeAcJIyRYGkBVrmNAYDLGBKkmaioHbNYt+YMisPFh6gPleYeoal3aPrR5hxQevqLAiPF3s15XRNB/ZGw97XkkXrJU3LrtdyHahCOjmCFNOKRTToRrlWeKRsuSrUQvB0G3yWJXWbSHxakdSWCe2Uku8i3Pu1TUN5hBsL0/DK+VbJA7kMFZZHo7ZQ7IkGg/G/tZQZjIeb293Ojba5qyEXBTDFuCoIj6jTGPRA95kGtpHBVQD9YzVlHPNica0KWtWWUvcay3Sj6YCbHnzumWFSPPd5TKQkW+OYZbjE09hZI7vGRs/pbDhvy+Jim1MtwOMlBn1xNEIDJGSlKiCk8PGva/j06zsw+jbWikYImgpm8yZ0EznhS+r0vRTnSWfheg0ibpOSEQYGdh9zitbObF8vx2Du60qY9DBb1/0GnBmbsleN5fnc3Zs2E11bZjyhEpuj7lkHh1OURWCbq9UzdSU9js6xMtgfTSe2cvVJjqMQO2jDx+51XN/QouO67f6wysdgxZmske0eNDVWIhI8wtHimjasAdf2di5EuqgYfVS7HSy9RbIK8CxRNUq73QQptNiHVO27Yqg7QJelcq+4/s3rruLgDLwz78LTQ/GddjUx7ElnMIaSM0MRZsNDlhQCj9KKyi1un3IsAmwZTWOgyAlohCKLCRkiR+liUoLdz/1doHVkL3A0jwWB7IbpIYOhXyFjrOphQBklPKhm0yk6PuVzX68dsmEq6LuaDGWmocI7k6RLXy8drdEqnMyuBrQ3GB5zHtMl5VQSTZh46rB6WnLKn3FRrjvqx8jowlYT7+VboopemIpoDcYAJMi4pcRZdEqeS/F3DcJkfNF7j1yKDZVq8u9h/+AQFRClBGdcX/A/R246th9kMZrlxdb2s+GWNqv9DwnN3NY/zWeb/57L9kr60/BaxVRJUh9dPBIZLzBDKjMaM+xe5Ww8hphGVAWDhu0OdQz3eQeib2/nBq4WJc5qkA+Ndeez/tzPUVAWGsWhN/IG7jH2PM6FtZcdFWJqLEztbDnmVbR4FDKN9asiL1YerPFFoQULDfgiIhlOrQd2FnEKRYoCxDDo+/3+qJm7t0qJSEdcH4f6xYX8ZV1ieLwmnZxt8MWEj5/m7W3M+x1j6wazR83YlFh9LYmkuxnBKUawY211ODwzpUXYIJIi9+GOoibOfm3+CRC74rWxuI9Zt0uGzymqfPhk0FJwJ/BPQGIGFt9cUpQcsRBRlJaKCUPy2oVIDFfWVPtgeKUQa4rnV6SisExFiA9IfMBhjBECyVCYaOJWJtFYmGnFiEiYMrUCyzb+Dr2GWzA4TAQbeSmJyvFPBz6Scy3EAwVGpPwlFTqJ7Y6CLKkZE6aI7SvNCJC/bQHbW1a0Z3CjenjX1aKaM/qGM9qSwVzrWDOHTLMT1orTmm+ciTcxvNQU5AqZcT8Q37mzbB8Y7wbRyWZi+X0pWWwJyPTcQYqtBj+81M1OHZ4JbeS53700R+/rI9ZTWXlPgQW+9jLTjTT31t6z6dn7fNIY6svAPB2wHzBSphLG8U+RU1auT9JUv7DsIGuDtTOvCsoKD+M1HByWZGe6RCR7/rk2XDU/S3iDpL6X28VzBxtvG2AKDsaDKGHe5Sv/3HW9A1llZO2rrh2+CWu29TZzU2ESqpaopzN34U1T+Ocx5JsgY9wZHzqtI3k5OeWuq7t4e1Spaim1qdB24Szfevlm3vkHiYdCzfYPAAA=', 'research_grasping/representations/__init__.py': 'H4sIAAAAAAACAwXBQQqAMAwEwLuvCDmLL/EToS6y0DYhDaK/d0ZVT2/Wd8EbnY0liUgszLKiT7F5yQ0fqPwkkoPFB+tQ1e0H30Y5aj4AAAA=', 'research_grasping/representations/geometry.py': 'H4sIAAAAAAACA61XWY/bNhB+96+YGmggNbKyXqd92I2LoEibpy6CIMiLYRhcaWyxK5EMRdneFP3vHR46HCvO9hD0IJNzzzeHp9Ppe8wM33LME9C447VBjTm8f/vL7A3sUFZo9GMK7/gRS8hQGI01FGyPwIXBHWrIpNQ5F8xgnU6n08mEV0pqA6Kp1COwGoSaTCY5bmGTsZLfa2a4FNFDAkYzUW+lruKbCdDzAEsiTlnNtGaPliI3jwqX21IyEzuSjuWUtDseYeFbiNyHU5HWBVMI3y0hWiSwiLsbqUFIY4XyessFNxg9xCkry2iMhs6zUtZEs7peJ7C6SoDe+fqUVqUlBabcpTkakgavlnDlCILD9tGM1wgfWdngr1pLHU252FOcchtgzUXNM6gYfR2nI/50jg/8epnAy4t+9XH/tn8d7WrR+zlw9UmedEKCC5qy1wu+oUTcLNatc19aoNMP8Bp0Yg/xEaNFnAAzslzOcfZTfOKfZ+gMOo2/Jr75gPXb5vdgq5rawD2C5juet06gabSAIY4Dzu9Z9qC0/IMqK8pRmSKxVAepy3yz1bKiOqhQs4Tyetw4gqU1X2yDNVNblU64D1/kWEFJQkSdgPK12JddfAtOCvAa2JGzEmzVhnJ0qCcbKeRf1t+ZRd6x/LS2ggdjdZWnIueVBd11m4eo8wl+hqsnocNTEz+FcYchuvsEGm+HkOIzahkNEZzH8Ayi3KnwX1RanebYi3Bxqr2QTJZNJTa1odREUZPAPhCRh4EkQKWW5R5tdM64vDx3IQXWUYkiamJ60g/0Omk+jCTPif0B8pX1I4E7YlgPURMIX4Np4U8ofx5+2UrzygKgWjC1CBjNXYedd54aTlHjFXOxO8cPMJEH4Lj43RLUCy7yWbDSiwCmEe7Y3b9ElTpFVWvWGKzUKayU722r+doeLcY6mgqd7BLc8KgoKjTbPA/cHRfBswA5spZsjBTMujzEwww5okJWcocCZWNhY1le01Dx2W/23sdtQ7Y4eCjqOdexg4xgIiDbgd7xrm7oem1RHPhX7nJNtwM9/pBMINIXYxfXIwBrCHidhhZGNvEbIzeKmazo8ByO+xEqbZsj2PPPmACv2A7Dt+Nz3z3afmeqRVTYDkyhZbMrgG0p0kjgRcJvhnVN4LuFTEulMH+hWJ5TMkKRBquj2TyZzeN04qHQqV5GBfJdYZIDz6m+baMzBUIjgpTOePrOJC0mwtzCTlOcvS9eHDFJNStxa9qFBvMdemAXrNwCC55Y8FoFXPdbjncubd0exXQI6Bmm2Qjd5tLOQv32QDx9ANyppGNpzwcJOl8JBtUzHOynVXR9aT9QI3sB+yeLE3vCYsEuLE5FvymFk8PZiTwnkudUPWLPrgr4fnB9quzs5kmDzIOnYkoR0kNT+TQ2f9TpEFF2iFAfYaGPSLsI9wmNlK3iK+oTS5hdpT/2oXrW372y3s9g9Hp+gXXuWIsR1k8XtH7qtB6+xjm/wOmUnuhsV6q6Kc3Xmij1hlApVJRDhpV8WAemkqZaRErkg++Xz50KapuD1kV1aKVEAymu+Xhhba+0k4HWiTrTXBmp6zCx+sb3keNhRvMSRU69Ae7lEfBoOw91ANctJEHCTtZM7pnmTGT2PxO9qjHUubjp97P/dTb6kFGAr//zpGSGpFG0wBxkOzYXb06nZgjen528qY/C9MbqVUZZtNNyUS+v4qQnaiNEZCqtkIlohKYLnBdGv60wLQ90vvyNlTUG8r8mfwOKdAbsyg4AAA==', 'research_grasping/representations/dino.py': 'H4sIAAAAAAACA41XTY/bNhC9+1dMBRSQGq32K81hAx+SNNsESDdBErSHhSHQEm0xlkSWpLx2i/73zpDUh71ZIz6sLWo4fPNm5g03iqKPyp6JFuRqVYuWw2/v7z5ur87d1zWIdsU1bwsOnRHtGmpZsBredes1Pd0yfGHYilveGqlNNpvdSSjlQ1tLVqageSMth0KWPAUlik2N36Jhaw5Ws9aspG5SkBpqUaAHDqwouLKMzhMGFNdkwcts9rVCN6yuuQbTKVULbsDimtJcaVlwY3gJr9+8+ws8FGBticcXUpcGhDXjedksiqLZTDRKagsVM1UtlrOVlg0oZukBwrtP+DgYtl2j9sAMtGo2m5V8BaZiV7++yFei5jHtTG5mgJ8K5r3XzJvEiXvxIGzlfHrrTCrexpFeRgm5Xfnd9EGUsESiN8g+Yuc6rlmzLNkNrDLNWRlfXlw9h1+AvpIUllGUjJsdhKxTJbM8dl786ZrbTrf4quK7Uqy5sQjLB0K5yl1i81K0Mi4FEmel3qfAdwp/8jL3kaSYynYl1sPjijWiRruSbzGD86hQXQ8GWf6AjoEBFU7Nz4ioabFAUfFio6RorcuWLy1/QArGalHYet/XZUZZI78DOuTZsTksBJq5WFcWX46G5xA1WIJ1NjndOxOrEAK00hLdcUQUbK+iFPyv6ym5mgks0j9Z3fG3WksdR2F30xkLSw5+M1V0v7k/hvx7aJkwvmgSsqP1+ACqZyD7ZmQbJaPxMYpbXL2T9lZ2bRnATBl0+x2vj0LHWvi7wxPLEV2NpXiU6wR+msOL54RxWuk+BvfyaENWywes1ZN0TVL+5d2rM9wFjTANs0U1gokHBwTroOICqMHgCNzTPNK+A089WufqNGa37Sm8Vu8nnUsqMigNR6qDerzqrHwTKpt+/0E5cdv4jhQP3js7dyKpAa4eQ5pYxAfdHonWWJRGjK9BZRFLbDPkoKjSQyhUC9MqMFwxjSpR76PBX+JDwOPdUgh9PsGfkUGOsou+sS3LEQt27KQXU9/PLi8ml229n3/VHSdM2Cu5nww5TYb5LasN93noSyDkzpVubveKU/7+7XvzBh53ab92nW+Fjf679425OJVY573v/1LiRKFmdMkNADrkR1D5OC/OPqWRKAMlLo0/zMiw7p3Pg9SN1f5dwobXTxA3GnSG55MEH22XnVWdzUnsUY9zimJiMTDP2v0YABlla27jMET66bTxUomtQNqeb/jeUCq6dpCEfqnvlskSJ/bzxqxN9CPdNyrGkCK+Y240+FTRNcCn8smkZRanmhtRSca3rI4PZqIzCbOQ7zCLhc2t3CCLcUi5Ejuswy3BMuN4e+tt4c2HL765qPtQgJUL2LxEkpBxPxtYTZN7TzPi0ZUFb03k8bNDY+Cuaz7hTUNrtjcZjThXjh0SgABE6YJD6N+QaAMj5eAgQ832mGm8iQWQPq9ehZwqzPpMT4PK2lI01GNO7w/eoFwqfn+5oLfXpzI1AHHXsM+/vw5XsZAIxwr2jU9IaHC3mBvxD5/OSWGcpOEtMHYG1HP2UZV8RVkIR1O3gPm7Y5qHcw5KAbUPX9C9EVkYxh66fUA83wn27Opm0eOp4OfgEol56B9O8HCszUMJuOvBFmMjgV7ug1eK/SX+Lqkr6dJLBIxQR2Ue75AujdlwM8+J0enYrURZYiUEquNpgPODSs5qhorizXMk3PKg+lRsc4grOD/3MBO8bsYP42Non7UwlkbLnBIUo1Awi5I3zTD2O96c89409BXKwEUySI4HMJTgNTEd1qbFdwnPJkc+8zBPFeQTvRHSX9SE259zf4OAFlmJARRVnGR4jcW/7sofT6qX0474iGfaewgtFA99BpfjivM9Pk4PcQuau6DjAwYuFimM2aDCHX+fXSYTCQ+ahtGlPeggbQ0zG5RhJWUdhzepWwylozC4VmXMOOnpTYKKHr6jXXjpp8E8X6K/IZXqMIuNh0/PKkRyc7Xor71NRsPmpPrTQb57vNb7nllrUTrFxRqzOHPxfww1uZIFCtR9s8gaztqY7YSZXySz/wFCzp5v7Q4AAA==', 'research/phase3/dataset/manifest.lock.json': 'H4sIAAAAAAACA+2dW3MbubHH3/MpWPt0TtVm3QAat+yTrNV6L76V7dQm51RqqgE0LMYUqZCUHTmV734aslcmFZEc7eFWkUXKZUqaGWAGt/7pj2k0/vWHweCrWT7nC+re83Q2nIy/+tPgK/UNfPX1zanJ1TRzN+X3w1/PuVpD0d5mTrXYhJ4xumDZ5JrYYdIgB8jkTxlM0t85z2eS8H/l18HgXzeftye6YWl5npyedaeTWpm7Z1dvu5+nw9mcx51y3eRjl68ub/JaTDamC35YwkJzmvG8JXoymbwd8eB1pvGYy+DF50e8vfRzoa+mo3b1+Xx+OfvTo0dvh/Pzq/RNnlw8eifVMf5I797Ro4urv0/ypJt9yqv7XNxH8ynzo7419ehiUng0e9S/MHQ1P59MF8ryimdM03z+7eDZ1U+T08kgT8af23Pwc3vawf+0x/2Sw2iYeTy7qcPT0z8+/usf8XOTf8p/Pp8O09X8c5PfX2GD//pu8mE8G/B8QKNvvh5o0Pq/bx+ACl3Oab7yASbT4dvhmEZdnUwv6KZdXjz+6Z7zFzw7b2d/PbC5nj7V5zfSFvdkNzsnbV3LEGyoCl1WqItHnbEqRTElaRudg7IxA1dvPEBlY1MgaUYwLqTikEpNXzK/nE4yz2bS/l9yN8UGkx0FX5wpiixmyLVqglohG+WUUskHlw2ZAEi5hhRVcNo5BK/rl9zf8uSC59PrrtLFcHTdMpfWzfSeF3r38EKaUyp71l3cjrX2Bd8o7YzWXy8cgQhOI9x+qcWTymA07vOBv93eoLWCmIHZZHTbK36t0kElKf2XZ5GuQ2PpL12eXI1bwyqlNd6e/kBzns6Hb8/n3Uw6yFWzDVWalb9U5/n1bJhbW8nHTRf93Kdu2lW69mRa5MZzng2GY8nscspz6ZY0G7R64tm3g/FkLqcKX7J8jOej64HkNExTSVO+POiXjLo6/WxQPo39wc3vg6txPpeiLKa5oNmse/dWLh1fjUa3h+t0mFu9dJfUUspDze5eIuNNDneT2rU87p4djlulUCcdeXYzspfOzvmf86spd/SehiNKw9Fw3vrBfHrFC6UZjW6M9K8jZikHsd9XC4PptsduHkyfUn4zm48WLOSlPEHLZz6l4XhpILznMY0zL5iNz7ZxcD65mvH5ZFQGn6zkt4PWgcpgNry4Gn2yFHQpOfxzeHHz27eDz43x7KfT7wftyQfn0sbjyeCmXwx+NVILw+DtdCJmsg8XblL8++s1TDqdSBf+gablA03bDxcX0nqPR5TfraRRnyR7wKE+xThsAq2uoZ7sUcxeofJVGMPKWeWNhkQeEwotgiDHQCSBD+mYas1FIGVNxEDKgYu0nj3FV6sLR6csQOYCRpsQtDGaU5G+wMZpnywIhJwuRqmsayEBW+QEscS17JlPJqNe4NFRefBL4HEgj7KMIinOdmnjg3FH2OwmbNaMm73EzBpL2QMwU6rz2QWNuyfT4WX3Ok/5Q5kOxWh2L8+HUr+Xs+70anbeBsBK4vyWPPYBQb+lXAfOpP5V1hNSrupkyZDD4i2UqDVYHz3X6ooKGEykJB+WXcJgrYglB8aK2kiIFMIGgVQYirYFMSXPwQUf5A4hWwEWh5wQXGBKVYNA0gcfs60RhFEhSTdKJm0JUg7wDpKMlGAZW8aHqLYKKesUHCG1o5B6wEDaT2o9wLhuxtgr+igJT9L1bHY1604u0rBI28i9Zt0TGY/jt92zVraVCHtw+j3A14PLdNjo6lldfbElxlVRyZoYfUEGjiZkQVa16J3GChhqDBbRorWUSIk0qsFLY1YrMmg9tihEBhDwpWq9k6RgS6mQNPscPEdVQXSV9iVWcF4eQEApOs4FlwRuyq3F1nA65bcy3qf9ZvYAol8ilUIblFtmV0AT4pcv3CrHRFJGfQTZboKs77DaS4j1NbGbAXYynHY/TN7Oul+kX3Xfj66lhbvXPO/kxOWIxty9Wuwmd99APSz1PryGeliJDvxdVJ/K6gkuU5JzxWqnnddCD6flQDQmJqEYFucLWMwmqaRDhMQqMntSVUsSk9muB5c1oQSO2gBkIhFUQe7lgCIK0DwWpOJIpSwHvKLkvXQWyzkFCCom57cILu2CwSVMaa/vzgxa0NZuV3QJ/Y+s2tHXUL3G0X6+i+plT3twalymk6HkMHzbGuHFtPWJ1Vxaf/U+cGh9CQ6cO/dVTk/OUMoxoMgf8MVlHQNZMilaZog2gybNkVTMySKEgOSVACLWTI5DcWnDvJ7kV01BL2RJURWHxSTiYmJtM34kt7U6OXApUs6CHA/WyyGVSZNNKW6RM4LQZecGQBvBLh0J4ILfKmaii+GImR3FzL3DZj+xcq953IyRxz+++q57dfLmzdOzlfC495o9QMa9z33YoFiokp544IwBlUag6rNKiTOlGiw5w957kOYoTkcDGuWaisGx5YA1+BwBldEbZAhVzI6iCBf5UZFAwFlXFFtttHeoo85VY/CeLFZylZKrKplCjCpz2CoeRDeFpfkz+YqwwjOuuc1ZvV3POK/g6Bm3o6xYHDl7SYhFa9iDC39+/vyvG8Fw70X7QIZ7H/zA0bBQJz3ZgKawgmx9ZSwWwAbjLXmCHJTThkrNaFN0XuukSL4pq8AVZ0mDCcTr2eAYtffyN3s0RRQEewpK8mbmVMQwV/QZQQUrWsQ5IUM2hqtWOhi2IiXKNtng0d0RCkKGuMppWqFzZstO00Yd/dh2FQ2LI2c/2bBoD3vMPU0nF9S9nkthRlIF8hPzqHs2HL3rvp9O5ufSRro7XbdQ58EZ7MMM1YMLdeCTVj3rqyeMEhblk2cXTdDFVEOcdfWAqcQao1VoSzsfk7cxx1KDc4kcynUqh7QBRibFVGwoQIEoZFtYOoYvMUAV261TSVBU0szCRGUyGh8Vto6CqngB3/oFPNcjMbXTZrD7vepHbWBZqljv7r78D0Zt9/W+4DscX5ns6lxW39G0n9NbfW3rZng9EwPbnYzF8o6vJZ/W4KNrSTsdzW4+u7Px++vuJU9rS3Nz5HTKdNG5ro7agqE0mc9Hq1+1bD3/PUDf1st82GTcUnX2VXEIUTuVS9LRVpUssNfCtqSCM+iJcvQAySNFyxoKYjVFro/OeeFf3KDiBCrAOcaEhMYQWcdKEFnIBJFQqfiAxhq0WQdWyBkoO9bOeqjWqbJVcEJABWbVjJ51d45obdBtl6EY9PF10I4idFvDbi8Juy0TvhnAz1lqhjv59o+r4bvu9FwwNGo97+XkQ5Fu8f2I3k+mYkRu8H5SynA+fM+dgti9+Nid0rit3p2u5O+2s98D/G67yIdN3+3UZl/3dJGUvrhSKFbUIZOnWETIWjSOK4MLKvpilc01phxiczi3AXwypKIqsEm1Sv4pZMMWJT0qXSIVpVJkT+CjJuOs6FkXSIlaxIzRJOOkwxlJkNJ24eu0vqNawXmrl1Wr8SZsd+oUvT1One4ocrc01vaSuFuy2puBe5YnzQo2jl92j7F7ffJ8JT3XXrsHKFz7/IfNtXuqpq8rusg1XYIYZ6lqXcRmg7VJKW0zGkiCjOidZUXoKEXffPhUtlQqS4uB8ushhRQsxRhMKoqrCiaKFLXG18omGtJKKaxa1yLKMBiVSGstCpXYyz+7wQfkIbGRQB4C3cLqqGV/DzlrV530LYLTdgNXWDzOte4ote4bSXuJoPusZR+fwRNhlNxyJKXqXknbvZPvv4ht756M6KMM7lfDi87YDu2miH3/n6z2wgPxtxfv0B0VH1pzfQXXjVtKdq6KwGELniHFRF5UkbU6WpAjqkaNUY5DpsA+e2MAs6Aos9ow2xm0SLgYmWNUEV1xrlRtqVpEDmRMzZg5VAykU3BG+lBlXeQ2OqDJdnssC8H5VQ4q7aQ1y67wxmzZY8Vpf3Rm3FlnxgePrj31eXyw/e2Bv+Hb7rvJvDv5xxV1L3mchyNRY2tiWmxOsA8o21iIAwfWivrpiaUak1hMEDoEan4kSjGHkGyhnI1PxiVdiYytRppGZcGVzwTBRWSAUjfMA6pSTILoFHix/JEgVQGbDlqEVgZy4AVGtgarU01JIdT26kvnYj2YtOgbcw+WLqfDWRuqud9qXwjK3KWRveNdKU/q43ZxpK064mhXcbRq7OwndFZZyh5oacECbz66561bjcukM9+9/svT1WzZnGIf4LK5FAdOl1UV1Nc5UhnKmKzL3ipPRgSKMo4shuANm6oMCGigQhZGgBX+ZI8qg/KFks5uPV5ESmkbrApB9I4qsRSOVRWKJVbSHByXFtC8GOkUSXvnK7kW6zwlU7SLZYt4ERFncTmCueggc2flltV6YSZvq6Bx0Rzn7XaVMytH0X6CZqXV3Ewa3RrnJx6/pe50JG00zC1Q0moNs/H6PaDMxjIcNmNWVE9PwliOmTijh5SLsEMBBl1Rt5c21cdotQAiBwiiaMAmo6At8SWfXSjoN4WRKCSIkqsKW03RK44RUkEx9qCsb+vLPIRsVBaupeIpGqOVztG3N0nGqS0SRiSUQHNZwCAurxzWUUPcrhtDiPrIlR3lyqqRs5dUWWUlNzPl5Y/Pz05evnx61j07eXVyetK57uXp69dnb1ZSpUeKPeBKj1IcNllWVlBPtmjVQoBXqX4bjKm6eC5Bh0iWooPiotJK7L4NclRURYVYq7DChZrQiujZ4CTnVSnFaAGFFW4w2eBIGcFYyApStWRqc4hTFlyC0kK9JtUiFskvRiGsZYuMKjGfpaeDXPBxWbwAGLPsaWBFz2x7Qyb0Rwe5HUXL6qGzFi7vpT7LzWDeQcKstpebGfNkyjTvvhuOJ7PujbQqS6eZXMrPk+uVlOmVZg8406sch02aNVXUkzUhGFuD9s0pOlquGkJKOXELQuRiC3/kPCkMMSujaxUDXQxBtKgtWNykYxy2aBk2K08VUdo++KTkNjYIZUrSlFq0JcxQhGfZVKV0ENQ4Sa5ZALTVeEcttOudeHjhzhZNPjZ/vN9pqkyBOe57savgWTeS9hc960zoZvi8+MuL7vWkzrtfJtN3s+aC3b2QriSt93pMl08nazYDfEjSPUDRQ4pz2ETaXFN9l+mCYcTC7aUJBweZTYo1e2NJY0MIg6idioKrKmypOSYyCNrozE5FXA+mpKsmWzAUUVrKAVTMlRWB9yS/aEZy2gfHXH1t/muWMZg2s1dRo9bb2X9JeW/cMpNc9NrdWaYrenAhLN+Wg1xI/scVujsKpR6DaX/Z1MOmbkbUm3PpU+2BF/ze3jBdTiTb8+F89Que/gn3AE/9C3PYcNpUT333r42etZCoxLY1hAq2CkpU5hJQJEwoqApo5ZF9NiGpbFXbe8k4J43YcLIeTVAUk2cvhjnaiKAwRdYOqwlyF/SRklPZh+xMMiEjoaiYHHOWH5jT9nyqVXBOwHv7tbxyNYgCjHc8Dwxud8+KYI5bp+8qmzaOpf0l00Zz2kM6TdNw3j2+SmnEF9LZumfD8bB7fBMzonNdnq9WTv1T7oNw6l+aA9dNmyqqJ5uKjaqoql1RyWSXs8aEFIsO7FwL16eopGLAFZB2YpertiAyqAUINLgYRfbe/f+KtdrFACWb0vJSxRTpD1BYhyQ3RS1qKWvCxDZxbkFrA9iYyZVgyG41wIJWgLCwRDXcEVL2ToRAEYUQtx2m9rjoZ2fF08YRtcfaaaNZ7bOjkjxIi2o0mc3ajoHP+UP3lKkuOdet95L7LVnsxd5LDy/WoW/I1LfGemJMrLlWbCA3RNmirVK2ACjwJnsg27zpqrXRVgdZxI9VUBp9VKw3ofvWY8xjTBG50ctl1TYWdF47DoFTcqbt0yR4c8bmqEUDxZygTTFyixEYsUS9TQdu45dm95ZF1s1buCWsKasMxN/rFZVGPPJsZ3dv6j3G9hdsDzC9mwn3w8mzZ2evuscnT1cvKLr3mj1g1L3PfdgQWqiSvqtQoW1Xl2IlEikD1lVHWZFmo4z2qJgcENmKoFJJbYcmlZJAIzmxy4o3vGOiVDCYRM5GoAwcNAqowNeMhQ1zJhCoJI0FI2S51ImY4uzlcUoiq7bmaKeDiXcm7xpFVkIHHPjfzQlC41Ey7SpiFkfQemduns13kB6LRnEzHr4fzlog9cvmF3Hr+/1mcj0TYfXx43X3mFcrnwel3QOcPKg8h42ZHlXV288bW7RwH7Q8bgKhChEb4wtlHQvHHJxCaS3RHToAW8vgnNOUKaZMvCEIAlfjKwNUUi66kqIybBwnyjnkwMGTK1pbwVK1KdWQIRYVHZvQpuvMNveT0iAINMsvi9AHd2f3wZvt2W8j+Gw7HkI8rifaUez0GVH7iKM+RrWHi8OfXz1+8fLVi5fdyY+vXj49eX7W/fLjmx+6lz8+fbF6qVHPVPvg3NCvJAfu2bCuknriSCSKqaEpIuVA+VQzYxZCJc8tSKlu0sVnkUslBbkQXHPrlsRoHJqEZkPQBE86evnHBNlUYVCTW6GWiqpwLS3cd/UouRE4kUAihoyJ5ENVIpw2xOZ+oCt4bFvd3omR4NXyRFvbjPf3cwVHfQybsLOeDWvH0j6CaL0J3Yyg04n03R9oWj6QVPl3zJfd48mHUSdFKt0ZTefnnQJvV8Lowen3AEsPLtNhA6pndfWN75PRERmNVWfWCoJRFqNO3PYbik3tVMOEOUUjpt201ay+vSQqPop2snbDLu0WbLYxo2r7NWmXQgJF6EGUVKqUdIVUki25apDbW1O8bcuoao2UkoeteeDpYPzi/k1Li2V1QB2WN+I1ynu17TVL7jhdt6Og6jum9hFZfc1rzyBz3dmo8DTxVMbi6+vp1WVnsYVAfULScD/Imdn6kHMPSb8vAegeUqZjOLoe1dUTXkKMoljHoosmDl50VdsonpooMtUVXwmhRWgwyfigKAqHsqEMxorwQr3Bt6EAl+IcCOgCCr28UxWsUgadtkHkV3uRVdiKsqPavCaKUI29syLtKvjt7oFkvTJ6eTlThLDsN45S+O2GeEA47jq400HpeoylfYRWX7Paw3981PY++nlYZt3joXQa7p62ziEM/Odqz/E+afbBZ7xPOQ7cW3x1FfXdF4KDQMdlQvkBi24vX5QcSkk44ykgE5UcQP7GcoblvwETOSN5IuV5g594oLbpQwpBkFNKVKLFKrGJiUNWus0oCga0YE8XlxJmY5iVFuxpFF5R2GYAbufBL797Ah1x6YhWqMLC7hF6uxrKeHeM+7CrXuJrRtI+Imid8eyhlU6enD39NDd4+sPZ2euz1apo9ZX7oH9WP/2BK527FdM3HCowRY9eOQafLNrssQTvU3JKgCGmv7L1mUJIqaCRtlJaea1JgyfnN2yZJ1iwbctWy9FHMO29kfwzbZLPt42GSgkUfK021mwlv2xERNUMil2MYtm35kmnjHFol+fctA64ap88o0Xb/G6xG0Q5HdfH7qzO+Y+RtJeK5j8M5WaIvH5zdvJzty4O6j1X7AE07nnqw4bFbYX0hYTY67ZNKhVnxe5XiCh//kdH2RUSc20gZ2rh5dhVUyjYtoWD14Y4Ra46b9AcKgElkRrkYuBcqgNRFZyFLiI6UFABmEXIZA4tnFBbBBsrWm+LMpi92qq/m1Lqjr9bCIjL0ecEZH7by1HhyIMd5cGXwbKPHPhi+z7Zf/n82x/+/Yf/A7L9LpNCxQAA', 'research/phase4/automation/rgb_views.lock.json': 'H4sIAAAAAAACA1Wc245lx3Gm34XXvsjzwXetFiV7RqIIkgN5MDA28hBJFtjsoqub0lCG393fn1sYaGCYIrur9l4rMuI/REbmf37x6fWXt2WPD6/rx8enH0bI5Yt//iK5vqY7xc/ZR3SlrxLTKM2yhd3qXObGKSP4Yfx/jdm2W3WfdI7bp/kv/umLv7zYXz998c//+UV4vJ7H/7CP34/H+w/j06eX9fj9+Mm++Of/88UMbrl8fFp+97mbxT1rO2W57UecxWUbK/oUYx5j9BTnKflsb7G2Hfzma07Y1SWz2mqetk7aLdfaa67nzFgrz79qc+n03dOOfUe+zlzOK8fdV+QjeI9ZLJexBk9xdrGRTss7+ZT65Ov9mCOVWVIdyXkrvSZnZ5bDs55Q9RGnpzF3L8UpTLP4mo8LPPOYYa+258k9lOGjO8m52k/dM/fRdt4+ucVH+NFaCzFM36KvpWTfNjHP/LWFeFY83fJq5t2IhXik4U91s5Xqhov98BHjtHX40Lb4/lST5VkCz+VDjmHsNFifuGM73nom5qW22U8JXSH1PespfDEWIMxelpU6yo4K5FzRxihr82/V80e1WJ3bVQJOGMZe84xlKygW2bEiLCaP34Lbxqevs8LafCnxXynmFP22GNpoRjCC48XWbImwx6KnIDatrhm8hRpCI+LLV2fR3wwsnaf2hWc5dnrp3o9UK4lgsy6yQOnXbYRe9hmHII61Vjmj5dx4/kmy8gknl1LPdESy51VjY0nb4o2CvkCpFe+CVn6/VF6z7xyoBtalnzRLrL1tUqb5qGDyfisTlBXnTHnkoxWpJziCPH2dpGGNJEPfsxTSISQ32piznpib593zqaFbCy2emam3bl2xmKeukuKaoxMzx7PMtIiGH8FWstkiuU0h1e7JjjPMzdXLKfFGfii1+GtynjQIRjpEN0o4SyWcio/Nr5FTnV7BZZX4PBvkACnYTp45NX2Es0ExBUv98NbEs4eWfdlkVfR9EviRWKphPs3j+KlYQ93GI04+q/WbnT5HS2SmKydGaxYCX9V7rnnE1o0cMLMcx97khJEVgEr2gcdvxZ0v/v2fvnj3/svH+9dzzB5//OX7x/98e/n02T4+fHm8/u2xfvlZoFIL779XmZGqdcV4T3/a4PmjGfUQx+lrtOnMlLGdaAyfV0z83li3nBvfuIbbexrJH4AWO6UMAkpSswh8RlL+k8/kAw/In2UPIBKswtdp2eIWTiVC4FhoXrhajJuf0RLzrQSNbwVjewejmg88Wm0jRr6cTOYjwAdPcljZbVsykGb6s515SvC47h0YEftkAZ13LpVKZfYM3PCJvIegbSdwbA0j+1Zu/fC0Y9TZeqgLqGguBB4ixZIKpd6BA+qBZLQaKbetWJwJUoXims/8W7ciCJ5AclhzpUke1ONb8xswVOYCTpP02HVSk9QbH2ErdsDT9QJquwnK+tlcIasLT9/LaNFRB2150LWbi6dM1ymJBZZWryqgCuuqufBDI9S+tSwGwtWQgF0AOqXQgAzLFCdPc0ifNEJi5Un0KVzK5Gzg6epoq6XqgZepDGjeN7ejz4OE2BRzBmlJh3DcWd0pZIVPVAr7DbR31v64Er23Rg2T0DHtwAqVvE4jKsFFXh3s36RN7760xOLGGfUR1TkiDsDt1FPde7Hq/D9EOyi5ubcb3QXgr/GDpR8fdiY+cXgyhNjpRSLlS6xzCYBNDJQ58Ax5zJ4aqBD2WYkqyJ1EIf96J2QQijOlvVcsukDcG3CRU5iLcLpFchHNCrz5TBrEHEhPHt23DjQU310H6GFpN+6ixu7bchEM9421Jq95aKBswcWRbytoiF3LWeLyDBwQb1iacIUD6iuc1E7lGXpywfvsXORnUQSkFtltAd5pcDBv6CcBqlsyYR+rNeXgg2qE8LQGgu+6va8WxoJyIlVkpcRSYbXUYTTzgmvgIMC4QP8mFXoepJZA5eXt8S+v3396/Pnl43787sOv9vbp8a19fvAXP38YH+3xjW0hy1jwd0FvxGPQRc+T4KGNIFxDW2zP/5HksK0nqMHPSjme5ip/BA0/CYEMdCy5tRVBT+cK5NXcJGyZvEaGoL16WpAQkJync8FQYh7FkNZleXLb6oYucnEFLONFF7gDUtaAqOFPoKLAb+6aBaEpAltx5VCFpUVJGJBstbJeGwppE3ImQ3bjT8MWRFG9JfN5FGOo3rWAtvGsweGHqWl9hAEBKlCgrLHMDrkQjl+8Jrkb2hwksZ8jsNb77AQcj0TAGguRHGShp0AImDukeTqAAqviUBkTfi+LlYd+j8RfWbEWoBTJFYpUAeJizxBFS4Ba3uQvCcD/OQe+oNUg9g3zLNdJYhiHgE3UzkJA7SK+pSJ9nBSMSiGAqI5K4iFyWwB/QjgB9TC6JcobAI4FPkZCGAEgiLmBcc6TDpSIFhXR0Wvn2yDcjDKCbVLpZCbh3TGvQHFPIHxnVjL3CLcD+Avdi5QqwrfU+ExqOiHvUHWuD1ChR3Nrb6udGDWeH+JDz8yDyENHssKq4ICWtCfKWoeiEUCgM8LMUVuVkLEqE45i5cgACmkNKgo0YPUNrC5j7ADbCetPbGRWSCcvyC0Okgph2yhikGEc6BPSbsflBPmTCB5EJEQ8whQxXdGUNiIKwIPGgVmeiI8/xqchWkaIxLKNkli/YdWhlwYSPiNtERTgmoQCX5wLi4Q0jkVoDgKFCXifVhLJNobz/GFGjpFrHaI6Q1xwAOrc9SKk73YZdZ6y8wltl8BSBwEW3ulMlAC+g7qE8QwdgN4gly2zpDug+p4oi6ALU+iPwg0h1MY7OMMgUHIswXIK/iIMxTdCDfSiCiGdTTjLusjycb+9voApL9//8maPP72Nj99f4yMd5ag2a+AkAtvhnng1xEEkg6uUfaDCEyhMBDfrCq4g/zA9QnPBACLQTibEivJO8CcshQxrwwOmZ5FFcIaNekDPMQIxZMFPABgxDFtiHziEh6CI7iUONjICWgbDF5ICSCgJIQLng9uEpsedYWaDWU8Bt4qSjup1IXb8FD8IG+JcABbKkiKbFDi6GgJEe/KHFCGCpSJOyUhVNjqZj2BFJvlJzRYWpBwSPgMtkaTFWBLmrXSHsnFcVcblZGwVpU6xoaybxJZymFVCKiNVJl+NXEYao9NJRBR/aXx+AUVYafimoLIabCZlWsGUmzETBQeHGZbIxwWqB35okztVeEQY0S5lopn4IUxCRSui+9PMo5zprzog51yG4dEg6DAcHL9Bapaao8MWoORYXd42yzaicJBw2JS93ERIKm/5/JyH0q7WnfOGlqUxgWJERliIc9RKIptJ3EsJPXv0PuA/E9JJ1A5mUz7gVQXjQM4nmgEURL/r1VqCnxL+HtmIAAIyicSS3Pc7lXjpgfgQY/4MWCM7iCXFVF2DJHDiyEWoE4EDUps6BBttFXmR6LEhXbGgXGRtQTpwBNWFRpGeqpEPAJv0aqAGJnGhqw/2KKG10U0OURjHbBfPSIImKQN9PR0JdoF/UHwwlOwEOhFjiFIqo1IFKfIsWOYqitGLgDcoakkOj/+WNa65U94Y2XUQSKCyQ/Y5IT9cDJaQ6Kz1pCAQg+OybeNjPeKB0iyIxF1gduqiIMsaOYtbP0qmDbO5DtXgH3EeCEUU6MV2yAVTh1YoCR1FOEgKfoBcnp3ohglerIU5wPesqfRHBCAXsItEaf8dSV5+Gh8e799eP316+fj94yv76+MPkMHjq5ePOKD9+oi//fb/9VVys9SOXD+KDsmCwuoYVrkWHgxAPyL/TF7uyWqMKslGSMATEDNpARNyFfO3CuCzUQdYCwwoHDcB+wZsdJ46TVhl4mb37iSUegkQGna6qTUzBujQF2YTsEeeEyxk18mS7oZ0WFpUQkLMQbBWyfGFP4aysTBQhgqbGlW3h/LHAe4IoLOKEY1h1AhJmtSWyIj/CGwFMQfWU30kfGX1UhnFIRHXbFAX+Ur5Qk6wcBzNiQ6MJC/qcR31j8iz3grALEDeO8LoegocAnKXKgGdYG7jbdU78ehczDzr2DIGj/ciyyBu2RqXTkXLwENJsagkO8XNP8mmyI94abGEl1re+GYp9IxCoNTngs9wZ8URKS+qcec6B0w20BIyMqhTfBROmq2q9VGEBrw9Ad58jIecExGIsHeeMBUZemU/DCpnAjANVRygPsQPxUvt73U/xeIBWRsiGrGDAyXnNz8RJYZkjU9oPA880iA6REDl1QbYjm6IoBWfB/GA1qgU6gubWTClZmh8vGG67Qi/hLKYzY0QnlMQsGRfc4b1GnDdTuEfaOyO+AOSQ+pYJAqSRbzSUXGgLlnHMmYYrNwgTPhDqD2oEYlRzkkfj9UPlBNaWl0q8LbzE8IGQuTUvIrwt5ovvEWDHAc44cgMnvZEECGUgcqYIBi6hpUtGY7HEykWKTlWEVOKTnPUDvWADUP3UR0kJcBEMMjG1RziAOMC+BDbQ+Vjhm6Zwe94SmyJLDcOCFxG3BgICd8QkOQyeolszJAdfgtUxRUZ7Kom3roGnaQeEDK1diSgMczw/iLVqCQ4lGSiRsgENMLSQvIyvHwn7RaKu114eXv9aTy+/TxePn6wT7ifz2YfHn98+fDj43dvr59/sLdHeLx/9ldGwMESIzw9X2OsaTtIBjQiZclXhAW2jNjgl+7h/OSwhSxsPEHooUdGOhmslGtRAUGjHt8jLGwABGDtIT4wjCSlxCrOo4C/jnwI+u7btG3YIbLaA1tltI1xBLvBhF6N3FgKHLpYDbWotFCzZIASA08G6hclkaiMrAT1vXrDPia/KUMbsGOIfFGLPCjPA6kJ1epQWzPjKwpveft7vDY5EwFv8Cx4fYP6adgDYAyrStItAIVCO3kiHrEQnnC5qOWgKoWSEbk2smTsQux7pCkULVGAIMPb1iv6D/43ZxAPMwAQgBtI+OXW1dxEFz9QwoanSdtCrS2cXcEMgnXRsNOSJA5/AS7gDPoMGZ8EzxGs21Og8DH7GEbYGAoF+xoBbRVPkuEHQT3OACOKCyjmVObTLSQyBTpBSwFtMRgMqbRqlmcvl3rRDhTNwrPzQcAnvh1akSnDnrAw4Lo0GKyjcKoje8BwJ7jHEw9IAc409An0A4AjM/BXanWVhERB4FDOOKcGOVwFtfhcNBZqkOL2+IbjKH5U2bnARnlhsaDcLcUpEk5qKcBUUBmSTXw9dvVqBC65Jcx5gPtr7egV0bydboW0GDPKNwktVHSVokY1IgsF9+RrlE9KABXWHFpEXEI6TiouFQezQBqN0hfioGtKRcGMhq5El/h6xXlMT/El+oTAWH0+HLmxyfkmCwb6HHDDi7NADQQ7zIulA7y3Uqs1NadgBJ4EtTPRPRWnvofaNBkaVU0Bl5gea/wWyp44kz8sZKUCriAloaMwxJMrnURyaofDdXtTJcDHUWfUSFFqdieUAEo3qj2Ovq1J4PKbf333+Pr1bdkH4OXxDRrlR/73zy+ff3j8/sP4m+3HNy8/PWJ+pPwPbVyoKgcvsU4MsftN/Q7cK+lBnoN4WJbt1oRazCthgc3ampNDImtFdxNTxuqxAvxdEI+AoKyzoTsgckVsGKt8EqDl1GslCokSk2LrF+KXKd/IRkpwh8CDYCng0EgxoBgxYeQUzmHD1hQMWjWvpI4yIgIzpeqU1mkzIhpGvTstqJnj4IjIn/EuJQOf8uIYNcwt8QsB8oNF+bB+dyLIdfTb3Blvt4CXhE7HnKIfHeZ8bdyPaghjzH+hbSffBwPyOwPVP28nGIQ4iP3tkXKsJu6RJK8H3dc7ZK83lheH44g3sqhHH9BVeBME6U6X7gBFxd4y7GXIFR6kIrtIK4qqYidIWQK+GrSL3k8RyGpDChCprI/A1Ae0uUfB8dNIXQx+2+CnBWKALZIX9qcjlRziH4mB0kf+YfU8xk8Z6U6TWSGrC9TZITqVApLyyFe4tqhKavboKXLAgXWeLVHx4EECfvQUrACIDz1vtNRBtFw1jDZUy7+7mg743pHgIvHcE35o8BrxSN10dY1Ae7iVZSosK+lODFekHlBOgXVXK925qa4/5ZCWYZfxxw1no/7/0Ipkckb9Nl608miBV0AKkaZ4FYC0kRvaj+BRUVO4Rp6KUDWAnxRft+XDsyKW+Q+s90IBoCkQAQHhwjdKTwShohuAS5xFDIAsNVwgvEraKrX4wCSDTVTnAQ/QuSQo0XHkMlwGqGvjA4k5hNxO27DQwMjEFvHyjEUvG0t/KjAC+WBJjZATDZwQLnxur7wlbfALHUdvPnjtIzrs5bkJjpXCMi+0zrq7clAc1pQMdOrmq4wXHsoj99QDQi85VOUKATSPxLg/Yeab3z6+effdd3/4UvjRfA0BAQWnQ4paE19QpJ51KRKsVbjoJFz7pAjNN9F3JXXUxLktbgdH6rdI2eM99jkBNxgS59X3Bo4QPxAN1huilkgrfEgOS105YqNslfXE9TcxLpiNHHZNneiIH8HwAZKwLR9O3u/83AJLeJDUiRNixt02KeQFUSIDUNJAgXbf+/2lgp4lRxEXoK/3PCUZgksj0uh/JMNcF8VZWrB5wDSAADKJUgDotra08ajokg6yDiOXPBK2Y47E+ixNxOa468a1JzNZ/QlFbSg9UOgRgZ96KkMraVCPukmgcQYpEWq3PZnhEfRyvz5YMl0GcQPjRF67uz5FuQu0bpVfib0FnpkUlF0Gd5YsJ8rZ6rUOaj/GAJOBN4D4odD5CZJxxe3UqSP5kYme35AD5JGDLwgOKoqoPfUWqnJSepQdnyChNBFUx3eEOgRLPaJaq+q0aIcbttO619vaDk+x5PHFAOySFwuaitiyVbilEIwAaq+G/3Y8GBSACl0kFTab9QR/syqXv3RIPCiodupbn64tcXWus9rh3vUM8pIpZNvEKTbENiREslLw7rJbUj8h4uK0BU1xZF5ZHTzt0yBZMb74Esx79TMvfF3kvZ1mAFj0fC0lNERaUdvEOTYBCLozoQ2LtvFEfdG8UA9fjNK40qQXMAKVB1elu4dDJfGLqHLgAS8tN4viDpQ4WHMzC3nuGwC51M7uAbsvHVHw0HZnDkB6WE1bYtqzj9rcUIvgsK6I/DXB1I1pUdd9SCRp+iLlVUhPxPOzZVyQgBOjTjQSSmxXE1SCReYM0xd9goS0wYYLuR2zvmLSLqXXjsDFj//11Vf/+x8AhAzemANJHVQQVY5J25m84fPaajz8dLINCAV0Ms4F2UR1IS4GUuhuvCMu0f/+buVPeZMjmkC8I7tGM1yPX6RfL9qeDshW/k0b7N14yvH0mBFNhU9zUhlmOBWSOxQqH50b8GBG1eOmqGcPAiFH1O1lgcsEiC71I+VSkr73WATeJVAZEgxZ+y+x9KVJlmxF27kqaUoTTA3Ia4Qdma3qj9rw4nfqrBgGkLqGghAlPzcLHADOTIogj4KcUuMjFit0KEcev5Y7hFAQDW5ppgZR29SerhFprS2/xNeiNE718hSoFNR+BBL9iKL70EylC5lBfsR2r47bJtmRsk0YhaBTG9pINPUFF8zn2gnDx+iHNjtPP0kfUcg81t3LXMrksFJFtAr0IVG0Gk77Wlk+biywZByUf0vahz8sgFQ1zkqqlLWvyIDjVpiaGlGLzYcC1LKydURUft4WYEb8nNPWQGpgwVWVfGcU7WheJPGsILjVoJXhKz2rve50R0Co8xzatiKVtDPajXLVR9SGmUAdkZeZ1EmFBc6KE7/rTtAM1Cbr5GFWKIgX0jJokqFoG7gqFpJueOWN30bVg8j8r7aOYWZscZ1NHgiXtsDaxcKb3DPe3OBID/bdlndFtGpUC+WKVp2HSjHIUJusJKLQk3eERGBbr/4akNqX5gXkj2ShUdJpkd2UFku6yv0btbhxK/A/KEyNoVHTIC5zApVRTIg9nUjXJJ4GiDH56fSknosUCMQHZbiMksZvkQAdUGJpRl08lUXAAyPMmvN9RYYNhMF8ahgFNCJ6CbxImkxpWHMC03xSI6YhfPj1kM1yUINpTIMD3NPnvHz/+O3r58e7//hlPL62j+vlw+P9+HQ7sugUkmAeogc+AnUo9NhZny1GAGiir5BM3lDfVv8q8UqqS+wCjvvSJwkKpRLAbHfmAWYNWHzMWtfQQjUxYM55qWHL2rLcA57f0Bs/dLkPcGGZySKoHytq0VHRCJpITfChHh2xpDtwnCGpe4fY0L4Zcdb+7t3AztoL8FnbXthbjYdAsJLSZL224W1PBAXKH68JREqNT/wRz4KSvI39QD2msjT0whMh7DQhwZJTtlO+Pmwo+CTKvWvKALzmPSZME0gKK0/ua11/DYagI0gJaoO8iRGXfDDTGxPR1NqPEiAUsIIFrkgU4e4lvyU/SHEsobawUdpHk09YODBQXULoAj+TQ8f+qa+NyT3Qocpfex7XX6unneGCrh1zstxGpgJjkAhCBQOT2j2DqUa9zTDTGCCV5MGY2xdGdAwRFZWN80Lk6XGzZj8yixiwdgmfsQAF15UZ2MahuZfeNeWxy1OZBYl4Q2HIhqNf+PJ5kqcYtblPABEzpA5rBKsXENPDets14PqSeNYWv9fMIERJllvA6ATSYiT4OAEB6uAs0QZC4qrmo5mE6dT2vYs6sogJRqUMyT/DGvMiFkF1j55BKQ3k3w7qrvsBZLkJGzhKtGjXu9/uT9SuLCSE3d4HSpSbkW7b6HIIYoIhHWjlC0DVCt2sJBXGX6tnrdTKWJGM1UGGaVXMQOAiR46vQCBR7vhgpL9llMhwvB+pFfAVFFBZSvCtiUfTYBaGBCgZhBSSScR7aD1IPMQTlgnLVw2mZUm0pVrgKlhahNtPIW4VgvRpoPGdZmSlh9MaaExIUYNMvD5AIp13qE6wLUbHy9jYF0w+jPXj/cf/t8Xzb38QmhBoBAdqJx0NimoWEKrEwvtcYtRYgobmwBMkf8jaXcWE9YQc1pTqHbEEYTw2cfDse5XStJsVyHdQGXkFupHspH6A5AJojXUBf3HEgXWFnLTkiOrgVd7IcODGHLKkYavBGzWZ1IcvoaNnkfVkFbmLF0RGIezB6ZHvfNrQRKp6e7YzbkDiHodEouToswz4kOAsp4ip3IRX5HVOUS/WzUtB+Ca+T7Ma1TTtu5uo1AFAYgPqYc4RMHgS2tepWamwJdLcPQUo35m8Bp8GhHgM1YgqNzX20M6wGz8JdTsMLdlFQQAMCaXLW4y7CSxd0VTjKG0gE7LVSNfwpcswI4K6+qAsl5uwJfxctcOfk5oGXs7s2uiqubI95FVmxhlVpAUEBD7j1kTJScNS2Mrrv2SxtauaqjrmpV1wHljQpelpGKJoBLVDlZCyaWpqOILGF69V80JMEiB1pECCquIFfNMFJKR1hUWcTDHWDnoImrymTDdIiwsmy1e4fTlIKN1hAoAT1MTTXmOyQTmUGG4igPsnRFKMoLNySBSNOMsI4uz7FJ/2kjWNutS/573ObQTjr6khirMdTXYIMzQZVPfKKHgo71qXjFdpHeIA5sG5RunfueKLaVSkIbkIIkjTobUxm/iEgLEM/BGZoIE2xJAqGptB1WMeIF/tQ1/trZmE0aK+DBhDJG9outTg/cQobRhP+1tJ49JZjqJkTd06wWJ1525UGeiH7lsxa94Wr3FnpTTA52Ae4lx576690YrnGRr39nwO+ZZZN70Itc4ixZo053l4AcpuxL0Gfw/fAGAaxdc2QyYCFTBM6tI5sQKUJDR5//rBHv8y3vZfx5v+5aef7O1x0eWKk7g1a0NxNwwTemefqhluXBe8Go4ahLtEcihFpVcpmp1C90Ttwl4zWo6YFTIfJHogDGCgYclBbxJco1eoRqpGvQvJOJyC5oIrkiiNdaeo1L6mXEl+zP2M2CIUhzAJnw5foeuRWogF7MMioriwQTGYF08Q37vm1MNG4iZihmbIFV0z/EH2tZrUPM0JYpTfgq9JSZWiut5TsybtTq+nlY72wYnvVEuKnBh8X42Ix4lnR6unoLlhRFw5hsPNchqJ0G8W9rbUtjpBqFTSpqwKjEiFFRAmwNTLr+OPtnDitW/9YBeOGvSnaujC9CKQLUL8DJwJQj5ByeQqoaJ6vBc4oqkBOXj9eDUX7LlJpvEs4HA8jWMeOxhGwO5uO4KiRK+OLVTYMJ6axNbYIQhFOmIMUV0ZJe1Om6iSO5lzIAuii6iBfTEJ+IwEPSavfvzWbiURgMQ08lZJwSRXTUiHy8/ZB0ANTNx1FIQckkP7XHjjMu8c/0CeoMXRdueQfVhZaLEgIUE2DZIVf3uDmpJTPzAJsY5mM5L2NPFtVFUDxxwade/bzmOxsAbH7vzyUAf8zqqCmElDVa7EQoWrwQbW8txIpGSCMxI519b23VDUtiDyH6+FsJ3zbp073G6K6u8qJh3lCQNnbVFqb2fzzHFiY1EVAMVYk/VEHHhMCZpIfBlg27nqUPu/zLQ1S3WQG737GYEkf0+LDEloHf5Y2kfmxdF5rmFApG9IZ5STpkun2gDoNMQEeTw0BqzpaajGFZmHjHXzcev4RFBzGFFe7+476j5htyhfPCPJ38adntDYvzar1CengDVglCQky5BaIuQgDW+1xoWTt3E+f/ppfHz8/u3l58e3683+ut9e/gKofP3Dy4cPLz9/erz/5dMPL68f7ziKLDj2KZ4rtoh7Vc+cIqfI0Fg1YjSHBriHzagOimZkIVGRC7h38aVeqAOA0NYoCFbRU0mIWcxKnhqfQNotGAN4QvFBvTlTIDIRtaqHgTJZZg7tBMYd5DoIAdfDc7BwzB57B+Elyrlrbgk+0C43jpF11EDKLUuWaPOzQ66VhFffbmj4aWrOm9cwnRCpJHFW21qj7egOah5HCz3eiRaPjgFkeWwMWzGoT9vlA1MwJk8TyaTW0BpTJ1JSBTlAKcQjVfGcajI+g8VVF1VqIs6lTRZf0Uik0O0iwNMdqoWQOikwAjSE1tmjQhT3gAMJoa3kTn4sOHHdbTLNap2s8ydNR4pG1/C9aWIpSgpPFBUUZfWCQ6DYqobwAfKd9LCt14QtQFkPcDcULTSGF4sDJKiDgOhrMHKc8eS7laN5tCuVTaNKuMUV3MaCU9fYfLCyIQ412+k1dhx5zwriGByvkQ+FMwAq6nlaBwMR2kDe7GlrX0yDmxprwks5/BAUXx3pRp0tHZNBVJS7ew6TQRQjaQwWrxUxiBprQL5ngla1/Y+oAlE0sokY0bgtq5WKmKBc/7SKTopsQgy7gcmZtdQAABZTe3x5oY+CwCKp758yUKGeyqIG7pipZKwCq018uFOj407j1yCSoa0k7VGdxQBWXBaaPGjwnHxz2uolXOE2x5vX8Rvw2gEcQC65Xu8Bgdz4yq5RegofpayDbeRERIHVQwLpVMwdT0eAlAYCJEl9tchhJ+24E6E6NYOyPPJI5whYHqzDtRNIJn3e38/96eVZLxQ/gidrNND0idrCQD7hy2cGu3U2x2vjy2njNQvrl+Yr7Y6jfLlez+vbTwKRnx+/SY9v33313BJGd6DpURSaKW4NIZGqhmb77pq9P0ZSL/lpWWpKsFM7RJ0w8njXmpE30xeMEpR5kEpYsTHBDQ0vaTYns4onA3yQPfAQdXQmLpFAQtQq5ZAvsD7vjthasFzQKZTaEdu4Z0DDh6M9BsrSUKDqL2y4BhpGxeFBhCNbowcTno7apVPPQ6QWBfNO9Nk1cM8fI0wFmjb7PVKy1aYjbrepqwEQ7CZeHjxhLXSQKGp4DUYV+Gn4iEUqGt9GbBLmGpemwjS2dfvCQTOQG1Q/UQo5GpYXScUPjI1969oj9FpXsIoPDYSS2IKYcOjIzxMcFb1Vtf24AU09Dh6/kIiB4l3H1EhUAlRNSiFYdFqMPGgeI3+eIgMRHW+1dQpho3/JXU1s+Nh05g3/lxA3pBTogkKkvKFvvKqhIjwMeD8CI0VMz9aEIqbYd4+8wovCVzhKzefxfUprzGUt4HFVo5FCLi7dxgGVityLCPoOdJLzqEUNBcbZUA46U6r9S7V2dJRgmI4MERSEBi+/8nN7HOQCGFpWP6PomCmmH+oEhjbwGoP2gWCpjlkTlqrriq8UrnQYQYvKM0v2TzV3dTiJEtJZCgTUvPPlTqfL1GkpQeOXfFAcRE9KI4fn1KwYANTVCb+Eu3fqhIxSgT9UgqYayiG2OjOBn+PpdGIJr134VNys9CuOHYGB3W8ZXiMxziK3sM6EWIdhayr4FtIaRAbR0E0oYh03gK3hT6klfkDdGtW6enmNNYknScHPeOclyAUZdR1dLFSg0zTM0MkzTSfdztjdqcoYxt60xTVW0syBpv50dg7KBoI1Ay1TBIEEhJAWhYWCwmYKwpHfv9n4/Pjty8fXT4/v3l6WvY3Pr2iT715/fW4RJ6Wmpp50StO2tk+SjiYuiAOtixkB+LfUhDrz8Gh0BrmD0PisdFfs7sbLqN+nPT5TQvVg9dTVxCYmong02u5YoK7Z0SqRgpcBw+dtOJWy1fjBIHt0fdRZXDHnYgW7mhplDPwFtakmP+pxdvAacF7U5m04gcgRBa0xWdMIAzQDgTeZIWTNPW1mR90prC7FEWFuXoN0TF0TjNc66ahkwWBguLA8MF2nUIP6LhCpTiLATHf8XvMgdg/koGCRoUiiccFgLE0crsIzkms6vIWiGmg6vCHlBPQV7egfjVz1u9WcwT0IYd9Rz313ViNG2642O83puFkyTYh1cySCWjC4RsdS3aPLUycTyd0C2aivrafYbem0LDIP4Ze1Q+BKcMgcROXOOBleEBLUxOWU7MbX+Ck+iwGkvy9C4FuAc++8x24HXxLkCjCaWW0CgHjLqIAVhFa9RFdBcY2o4EfvDo8np/BrSRCvSQviRQkrASQlWFfTXkawO2Vb1DzDnWHMIqze+9NGyuJjyE2HJVnro5FHnbjm0ZQ8gjQNG6s9EhyY1TQbAHzlQFivtIkDPNFWKKm5Ag6haCykJOSNbP5QrrZE9WkAR4PEvGUjXvCyDMydXuodwrkHgYrOG8KQk1Rz0VbHyOqI0cD/qvMNPY5ZERn3QBogAqPcrSpxIZ8Ny8yh3pYmhjGA+EJ399g08aMjoaS4xtOzxuFDIuwSwPeABySFAy0LMUMsYQJwzCjWQglgcSl3lEnTpLs/HsDvSD7NHS9NTfm78wcKIhAL2MrzaHuCp9K2GuzXSQ/yMWv/cWr8ApvOai+NQWnejawvU3jyx/G2Hu8+fv7h9eOvj28/gyifP/yKSHn78On+8/Hlx7/8+vja3ljiz88/eQ8E/fQoj/NBs23z9fPnD88NIQchAQpJIIqwSoIzxF5DtWEcVGzOqZHnWOR82Vs2s89ds47UXFd7RyCR8VmdMIjXsXhkllwRYLngJvIbjEbSgj5FH9y1h6skuWaQBULdknGa7kEDoqC1MdnBWpZU4ySTP0D0qa2lww+kIEuwPIFWf18qQiO+N4uANByMkSKamXQY1JYBdDWPMWVw9gZWp7azPdpLW3KA+T0IuPzd/z1Dx++ltnOTjaJqMAsaVcc7LJIEIQxXyU4LnQv+YeoY9/VE2wcwMOqUFjZ9iARn1ElZTdxqVg1lr9N31dcGouqMgidUSXOy69pznVYvOrE3nLYRAksjJtGEpMaHQx7a4F9Op1l80vhj01TkCqBqmHqKKKTVcBNMotF5XhevtzRcyg8XbUZTa+STfK5OvOrcSgAiXcaJ3tmSpZOMkKUt5CvoOpuspABHzQRi0jVoL2+zdF7rzkrgWk5gqdzz3PgWVwxpuqMjC3hYzUajIBMeAwMsJTIohYCWI/fQm9gIJFmamOi1/bXbTq1r06ASLKKzdbflO2Vqd5M1Atp0PBMPrCejpij7sdG8pV9nhl+YKKzCI6CrJ9x20K4bUUjk9hg6ZaND52i7mHUCCEO7jvacSbn8nKKd8vPb2Qa1vYZ81DxLW6dvQESWYss5IluLlwoRNhoqog7K6FQZGoyL3+juQgpgk/i0IWHidD0BBlMHP1sDOkMRgZtaMiP4XZAaaPtnmSk6vSTHbyADJoie8Xv86J5VO4dNG5p4ry7ryBfOqJNRA5JJwd/xFHLGhOPqa051D3AtdaGL0B1FROuS5itCtozrOIhnR3A2ohmDPZ4j+l/ZJ6Djwf/8xy8vPz7e//C6Xj+Mz/b4+vWv294ev/sw/vL6Zvs5tf9u75fPL3+xh3f98ae/Pd6Pj7oq4e0OtqjtEKgmSAvnprmg7LQbREEN0FQ7dvwZ9kqjZoi2eXSisxJe1MLFDSKmeUPWjvyjmMwlnY3SiXWwnYKA4JDiCArs0dL+tPqPKB3wio9qz2mwUMfUagAfc2jPUucYyRiYIQSdJNcZGn5+HI28bJyScp7KJ9sleyQQkVW4N4q1am5IOyHIbVwbsjuoD8eqnzsiKiOmE+7oZfiwZRjg6T6AsyYXqulTB+6DheFe8oBP2tp70onj3LGweD4yuunahhE7qX4VN8oDTcqCYQybTtRoVvtorOMeYtb5YkkZXXKCrABOWFYdiwsIvO3Sc3SeJ0YO8eEkYlxq96lNXLQqvmn2vBKjBl37jL1DqelUovog086daQVKx446eEh56DDXhgcPpoFlLDlrjDc7vDoEgJKgiJwmzmCPYHjzu2+zLBkaXXNsO2Ipq9QFca8aMVJKCJZCxHFImc7coyZxeFmLdwv03rGAL5VX2YiZ5e54MMxyNzqOQ2XoXBp/tNSgtaMTVxHzz6+RF/7OpEmSsW6BcgDvJkw+5KrgsVO31k8d3I69l082bNLWKJPGkZBmiF0taog6285auHNnM5MOjALiucnyiYD8VvrvCIqmGchA2GqbZvOf7gPQGNpt0RwSa86bEogeixrm7e5Fb7WcYIatph+5oAtRsIUQRJi3V8vya87jVB12aNqg1S5ZU1MEsQiYHa+LUDb8mwU7WEtEacVDVXW57iYtdTajxpxZ+oZ8vCPcOAOSCm7GYiO2MgIsVEoFQdw1dR6KRQCx3o+gpnU5k7Qu8oCFMhJEM5JDR7uB0FG6dgV0PMmhr7wALWtW0jTpeyfj/vRvf3p8+3o+P/78+vbjJ2HJ408/20dA59uP4+c/vD53kVC2xIDiWjqnwfJj7dDpS50SPIvOX0hHrHabkDAGKgDldXTyKxLSe5TpjDLkotV65U/rwiiiInTMjqQe+gyNjIV70IyyN+olIFcjHrLcPr36e27yA2u4FGIiz9E//GtYus8I0LHOgpLgp+kYTIfZeemkA1szaOHWViQ01YSCb9pPhKfQHNAoXHV0AQv5jg2Wbiz8EEC4pRSRDqzkPWin09RTB7khLwQttkMKl6T3e+FkTFsnqWJvWAntd2t6gCXUkmR3J0TRGnOPQOLrmCj6RRtlFWWirXHt8jagcKtrgugISl/dy6MpUDzEEx/zVuMP05M0EEn6+TsiBhSBUjFN2LgdbWJNHTo6ZcRdNBo3TW3Lp6dL8lfUQAZ4gB8qEEvk1RrVVVEssBXwyIdjYF0JUB5mQDffCDVvOM0bpjfCqoSpqM+Kk4maPNIpNe0JUpea3wTcNGl3tEPdtoY+qZx18bGjLbS/CSVSPuEKHh1dBCRSkU02HDLiE+2qEUJD0iSH6mDh57M5Sto77TmCx3B7Ud9f035e24xNx8/UWibxNS/e88AYLBuqP/gC2aKnQPBhX7RPhzxYCtRZZI9u7PDSS1U3CjXebOpssbYwnLvNbjX1+j1NDAVqniXkPbW5DSMYRR90GRlAXTW8UZZrVyXdeZuhaXqyvx5dC6G8WF3iENkIiCWUOAoUytNJ1aOpEIJW1OhWtmPmAHqNUfr76SzsnSnlk8FONAxqVKpPh+RMZ9VAGn5Td5QsGIcXBxnTgj0wfLCA6fIF1NWlm0xa8h9HJ3Njx5wZJKbjOGdo7A7JGbRTSVrqGKeT0dblKfhHuZMLK2/z5fPjN7/M+cF+evn4GeXy8eXxm2uRcEzr80UVDehtwHEUOd98mq+aJ9e5Pd01AoTymPCgrtRCepWgM59roxwdOuxyvI7oSb+Hk+t9PSBdNZgDVjFoKEjmFKoNCPcByMCMR8Zbwvwe99AWkobLWgkaVp2a+USiLO/uaVndLpZ0ZqzILVCVpUAmuoIu6GqV6z8Tqmppg0PTqshfoLecY5NcLEM/nHWyALAqDvijjjVLq4E9hBUZdYdXs0QohK/7ItQKwdzrQKNwEp2ieXXAYddeNbeh+xHgZ1SRrnAgO9PtCGzd51Q7Jaq2nBoJeMF2RIvdNLTp2t1jrVQVQawYcF2zETWeeLf6Nf0qgQ1M4cRG4VGxAEHSZGogL2k/qGVdR7A1Q8+ngkxZJ9ESQPTcI8AvzTkWuKg2Vmwag5DMOvCUvBqWTAO4gCP29XgojZylTA0Rl9vzJFEhdePfI+Z0QZSm0gEz7AVRb5r2yVtGGj61ve+hfAKP8at3+4lAYoKObmVQMgu/Iiqq8c4YcE0i68opQGqjhVC0CHLU4iYJYbF0h1d5HjLOa4IF3wYla0ehVTxPWlI3vTZtlZNcabqA7CmTIGGtIoq7l6uDcfFQGDq3aHixVjHW0WAeC6FLgchz29JY4H7W+IdGsI92qkULtwnZEPKCXJYIGoLIoIpIuPMKurtjNR3JhAshp6VRDn7BDy8jK+V17yjQAIumcCJ6vIZ073QLSMk7O8ba6uB98RpdGTrlPnlj3Sejpunc94q4O0VJtc+Db9yawJKIgkDHlHXXMRVXNSiqs/saFNtSLhSZ2jDtHuoNKhGr6kWDNb100QwMmWEJ2I58mTpeTRUldQA031vnyHGlRESed658/a9fffnu66//8OXjj+++eff+HUjy9ftvv/3yO6GJtp/0aIAwYCiklWMLsZO18NTW7mDTYWkIXekVdG9huds9UMO9capq0gyrp90M3jg1d2ArFb3uLQr8YzpWe1BHiA2IWJ8bgwBo6OaK26JMQmM/pnrbpr4kbKbrnvgjLALiCbzVmXFwLu2gnp3UrPYGdMhdH+G3TgLcqyF1qQ2VDKkjhhDIc5pm2e71dLruBGQcuuwhwN0whFTZLcJ751ueug2OQh1VssBjsZYvunlCV1dpoIdUi8SM94WOdJPP7Mrudk/yJ91zVFDEDbtXNMNRU9G0iib1XWlrT+1BJfISKgL0ZB8SnlFNmDsglbA2vVXeX72WhDE1TL2QWaPgCfjkS3R4xfV7zx3GNaufhYCgGG6vVWHsurYI4FLyHG3iIjjUdOoTPGCph6MANRRkWUdqus7lYm96unQ2yz36UuMomhDtHeure/HUDeo6cKtNavTFcVIG4c718x2aLkAHPW+7wGeSjFiGCBgBWfzo0eSYQ2gNr3mZe9vlNB5LNIgGmxQpeL1xdv55N1zlN6DciJROuoAPYS8DjPu8Uxng5lGPQ+di8F9qwB6IAyUKcod7okctOtxnl+GCsnRgmmqdSaMcVoQHGrtOi3ef93I3/ntV5Bu45MV1KL2tE5WraASB5YEuTQfj+1g4F52oNJ0pX5CkNnO6dVZJIy9IkX5HudQFBvLsgB+qsEU5ocoQB8AYKQYchA4sartk9USciViFmwHL0Oqd+kn8Liq1ayZNwwHT6xxw1oIUODM3yzrMPRzYse7JPJ1XBwZ1yUtTpRKuxmpgtjC0YWtom0QnICYZiOPS2WYEO2YuHd0JMjRYZ0ep2ly4g3PfjL9hc97NXz99+uXT491P82Xb//389sp//H78pJtY/si///0yW3yyIVDOvWdK7b2AbNZ9A5iPkXR2vemkLkWEINJRUGIzBaLwT/aXC7LMwBnaP6y6RaTqQCWUqONTUqvaRhzedNafUkC96SoBdH3UdPM9kQF8z5Uov9buNZU6Wer9uWftdBjYs4YnYCZNA1XmtyZPok/I/rRvN0Kb9djLDFIIJuPU9ZP3HOqiKE26fq5mum0HmCtdc+itQ4q8LsbrGjmqYd/xZwoPqHd5k5A6tyBa1e9DHuoAxBF0F4iuNNR1XcZCoLFUT1QZtDd181NJy2vOBg+cnceHk19un9SBHq+2Y9UNNVnDVPPwLvnA4PfYdCTj/IWbIKeDv9fNRzoJrpNt6A18LvZJ++wARNAcsG4HSDqsu+8uJEgMrrQMugGsEygqum7WLU3YlzWM/8XaAU+IZX4pSPffE8FBJ+KfM1t++bu7V5xG3KVCdRAgSxtnXW6jdpJmOHl65GXDzqOlW0BOpHE7unw17lCiOyLdSHNdLalrhcOZWAP1i9a9NxkTiZPVxBAaD97is0J/Ti10nQZF1hcNZA4cgTaBgNyTbGi8HxOjDZ4lk4yIuX1jkg3I04at0B79e0dneyzC5KntWP5OvUI10nWDWbaFcoMLa9fkJmJsAxzgxrPHDpBXDRpqokrHL+US8T6wbIVV3aZ4sUa6bEc37SCTetOlKLoFR0ctzt0sCJot0R2IfOO9xqfq4h+yWPcNHnlpQqhrZ2tEBcM+DrOFDQJqntNPPNCBww5UDdDOpEPRlnQXHCZ3VV0Cpi1DzWOfAJuSgLpCWQs0VJgKpzr6sBWZq3tEQ5rEQTc6aq5LuwemTqmpxz0xoxSB7mQ0XHv2sPq9d/K7H97s0w+vH/Y/3I7wnY2fXz8//vzDy+cLKrpVUgMzRxkBJMtLg93UQ1RcqqZt6wRAMBW4V00KGLDZNd2pe1nvpoR2kHWPRNGNWTCZ7gShqv3BiqJW0b313jIHKxzN82K7TTfbujXiyXdDVEPFullMJ3aQpHDR2NriJOFG1xHGgfJfhJC/R8zqTuGimyOh1Xovlj5QbkYc6KIO3bozj+73JphIyxluZTkB1cAOqcuwED5FO4Jbk5X3vNqWByhBrrVp4Nr1rbGNcDtzOpFusS1gCMDJ5X7EGLocFAgIfd3dFfB5H11SNqUHoq6e1sXaiq2OsAdd3Dl1kHbGDJ/wOXWEQLqSn9T13UfE6qck9V39QoRtHZBfJRV1AremoHrCyDlsBlge0K5QLvaX1UO43AmijpLRhhLypVRSm2oZWRMQuvAu6aYqdGq4dyNpw1x7ylAsstxvpE0QqEyYu5EIUAx/y1vqjsU28AMAltcpU+39kv86nyIWiDrdtkAsdWXuoqJaPXp3a9ALDZh07w4eKvSjA4VRDhFkQU9oSPPAlxoYpB60daMr1244I9Uj9wGbSco7sQ4sWwbodfqJruum8rp0SEOtJB0g8Ahjln3cjnGEhAHmmAf837SLuXQzHNIUT+7BDpSdGrkebML/bw2YaGpB0xXg+YU2U6dOtwsXCBQa0xZf0HVnFAALUEaFycAqXWqsawHIKjTDknCGms5dEXevBnf3ELp2/1HCgdrrgu2hCyN1WhSHllHaneWe6sprdjhZvs02HQ9eILOaRvwp2krbmRNXIAE/5FTu9TdNw9uwJ9afPNftRPBqu3cx+IyOAzeTvi5guNCCusCNhdURG9ynLs8D9j0OAXZFKWa8o+5EIEHc+OLf/+u//hvZcWvm0V8AAA==', 'research/phase4/embedding_experiment/development-images.zip': '...', 'research/phase4/embedding_experiment/dinov2/global_embeddings.npz': '...', 'tools/research/phase4/download_models.py': 'H4sIAAAAAAACA6VUTW/UMBC951dYe3EWJaYtpYJFOVR0q1ZAQZRbVVneeLIxjT+wnS1otf+dcZJtKRUn9hCtJ56ZN++9yWw2W/50napVJNLem84KSRpvNbFNg1HRkYtzoq2EjlyehXfEWIK3wQQgoq7BRWFqINaTaO/AENtH10c2m80ypZ31kQi/dsIHKEgrQtupVUG+B2sKYkNBPGRDMydiekWmnC94zLLLs+tqS6UydnNEF7QRNaysvXs5RsqgRdfRYrzw6tmFV+VGxXB4UjoP0Qtlym4jD0/evNV0l2WZhIZojObzRUbw56o9UHbq170GE7+kk88lhNorF5U1FefS1pzPxxQmpORiup3Tshx4okXdWqQoVDhA4eFHrzzI6pvvYf7ueY6HjQpYmz69WrTQuYo2fdeR44OyboUXdQT/qEtttUbVri9O6b/woBq0iL8cVInRv7EQUTk2jJySQj5WUQ1qHFEZlnprEes29/TmoHwryuZ2e3ywo4Vge9TzBbYF763P6T5GdB8iWaFBDIGfiPrpAC3GJNRKP8ww4R+c0PbrtTLrpCVv+wdHBCNcaG3ke5MOGc+iuQdnuZKJ+hvBBj1uiz2w6hF30dladFwqj0GkqRjsW9nAwGyUt4atASm8OOffPn9YXtF5gWaz9xyNijOYUN3Q2ppGrVkyM7oQXea8RdWD9fzpqwEFC6KBiHtjfcDY1+Xp2acl0xL/f7x8v7y6Xr6gtyMPaU/QPdvdyArulhEaiDIk/5+mk9GHmuPQL1Pdh+AkfMNU4I3qADeD4N7gop/j6crGc9sbuRy0bqhWIaBOZJtq7CYFB/TVtOcstOLo9Un++OpexRbrWwcG3bKicyICCbb3NTxi28+8QoXu0tAKCc87oVdSLMbLqCJKfXhwdPwiPebFiuJ0pGW9kyJCPqT+gWjg8yYBva1ahv6Tag0hTsicV7gtiTYme+1Cvh3JQxvRxRMfPTicLv5wEh3npIuxz65AtiO/g19h3LN5liG1nKf+nFcV5Tx9eDini+kLlP0GuYbdaYYFAAA=', 'tools/research/phase4/run_experiments.py': 'H4sIAAAAAAACA5VVy07cMBTd5yuuZpOJlMmCPlRRZUEBVWopoMKuqixPfGdiyMSu7QyPqv/e67xDAxWr2Pd5fHyus1gsvlcluByBVy5XRj6igA1yVxlc4b0zPHNSlWAd3yKoTR0qcI+F0jssHZwcfU4Wi0Ugd1oZB9xsNTcWu/2NVWWwMWoHmru8kGtoHZe0bRwGLXKT5WxruNWy3CY65xbfJoqMaAmCU6ZLM1XJBlgx4L03s3VVigJj2PNCCu6QCcwKTplyj0GgIe2BJUdmW3nkl35nlgJtZqT21VLGhMoYiwKdcCEYbyOX4Wp1o9ZhDO5BY+qRx4T6VyUNivTaVBh9hH9T7pS5fSlnpktzjNc2IpayW61k6V7ZLlPlRm6nSTkWOg1H9E1u+8vVxXk4V2qnBBZUKcuVzNCmP0KlscwKqcM4FLJU+4Nu8Sb8OVfB4F5auoVw9owEgurO+9bcZfnKknS7oxAVcy3uUG5zZ1c25wfv3s8Xazh5MUQbRUe0yvRRzxAy4WNEIzFRtzE7L+0HzwcnieqkVqgvY5dRkJFpTs9LP1NJobiwS540eBODXDBHk7GMogjkBjoPYGERfv8JPCA/B0m9oInKki0S1BboBF8UUIUmIYUp1kMwXFLFqwfrcHd6L6nEp7OL46+nJ4cwjgSiyalMFcBLAbIUSIIQXkRck4tO1guU+tX6YbcU1mMsnoAs/G10Kqmj+s2TQDZoKWiEU4e3yyG4FRUdPtOVr14riXkl1Qmjbc3oeG+hVA7OVYkNw23JIYTKfqh59HGj41F7b+kgHtK1ozHK9FNU8zXMQ88S3El6oSsH4+FsRzj6z1NK3kcsWV3fdm+plxBrPA3/dOiRbTmAjr2c+lcmbruyRv5pJ7V2H0M7Z4N/aoihn6Ah5KkphuZ20uYTw/Rm024RBdNfAo0EvdUx9aTnN66T4mdS42mH4erSYRkFkz/MtDoporbS/Bvpx/744tvl2en16Uk48v0F7YshJ2QHAAA='}
for name, value in payload.items():
    target=pathlib.Path(name); target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(gzip.decompress(base64.b64decode(value)))


In [ ]:
# Configuración. Añade 'dinov3' cuando tu cuenta tenga el acceso aprobado.
MODELS = ['dinov2']      # cualquiera de 'openclip', 'dinov2', 'dinov3'
REVISIONS = {}           # opcional, p. ej. {'dinov2': '<SHA de 40 caracteres>'}; por defecto, la revisión oficial actual
WORK_DIR = '/content/axisforge-work'      # pon una carpeta de Drive montado para reanudar entre sesiones
JOB_PATH = '/content/research/phase4/embedding_experiment/development-images.zip'
OUT_DIR = '/content/axisforge-bundles'
BATCH_SIZE = 16


In [ ]:
from pathlib import Path
if not Path(JOB_PATH).is_file():
    from google.colab import files
    print('Sube el ZIP del trabajo de imágenes preparado en local.')
    uploaded = files.upload()
    if len(uploaded) != 1: raise ValueError('Sube exactamente un ZIP')
    Path(JOB_PATH).write_bytes(next(iter(uploaded.values())))
from research_grasping.phase4.bundle_io import load_job
job = load_job(JOB_PATH)
print('Trabajo verificado', job['job_id'], '·', len(job['objects']), 'objetos')


In [ ]:
import os
OFFICIAL = {'dinov2': 'facebook/dinov2-small', 'dinov3': 'facebook/dinov3-vits16-pretrain-lvd1689m'}
unknown = sorted(set(MODELS) - {'openclip', 'dinov2', 'dinov3'})
if unknown or not MODELS: raise ValueError(f'MODELS admite openclip, dinov2 y dinov3; revisa {unknown or MODELS}')
STEPS = ('Pasos: (1) solicita acceso en https://huggingface.co/' + OFFICIAL['dinov3'] + ' con tu cuenta y espera la '
         'aprobación de Meta; (2) crea un token de lectura; (3) guárdalo como secreto HF_TOKEN de Colab con '
         '"Notebook access" activado. O quita dinov3 de MODELS.')

def secret_token():
    if os.getenv('HF_TOKEN'):
        return os.environ['HF_TOKEN']
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception as exc:  # secreto ausente, sin permiso del notebook, o fuera de Colab
        raise RuntimeError('No hay un HF_TOKEN disponible para DINOv3. ' + STEPS) from exc
    if not token:
        raise RuntimeError('El secreto HF_TOKEN está vacío. ' + STEPS)
    return token

if 'dinov3' in MODELS:
    os.environ['HF_TOKEN'] = secret_token()
    from huggingface_hub import hf_hub_download
    from huggingface_hub.errors import GatedRepoError, HfHubHTTPError
    try:
        hf_hub_download(OFFICIAL['dinov3'], 'config.json', token=os.environ['HF_TOKEN'],
                        local_dir='/content/models/dinov3-access-check')
    except GatedRepoError as exc:
        raise RuntimeError('Tu cuenta todavía no tiene acceso aprobado a DINOv3. ' + STEPS) from exc
    except HfHubHTTPError as exc:
        raise RuntimeError('Hugging Face rechazó el token para DINOv3 (' + type(exc).__name__ + '). ' + STEPS) from exc
    print('Acceso a DINOv3 comprobado.')


In [ ]:
import json, re, subprocess, sys
from pathlib import Path
import torch
from research_grasping.phase4.jobs import canonical, sha_file
from research_grasping.phase4.frozen_models import load_frozen
from research_grasping.phase4.orchestrator import atomic, run_extraction, export_bundle
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', device, '· torch', torch.__version__)

def official_checkpoint(kind):
    from huggingface_hub import HfApi
    repo = OFFICIAL[kind]
    token = os.getenv('HF_TOKEN') if kind == 'dinov3' else None
    pin_path = Path(WORK_DIR) / kind / 'model-source.json'
    revision = REVISIONS.get(kind)
    old = json.loads(pin_path.read_text()) if pin_path.exists() else None
    if old:
        if old['repo_id'] != repo or (revision and revision != old['revision']):
            raise ValueError('WORK_DIR ya tiene fijado otro modelo; usa un WORK_DIR nuevo')
        revision = old['revision']
    info = HfApi().model_info(repo, revision=revision, files_metadata=True, token=token)
    if not re.fullmatch('[a-f0-9]{40}', info.sha): raise ValueError('El Hub no resolvió una revisión exacta')
    weights = next(f for f in info.siblings if f.rfilename == 'model.safetensors')
    if weights.lfs is None or not re.fullmatch('[a-f0-9]{64}', weights.lfs.sha256):
        raise ValueError('Falta el digest oficial de los pesos')
    pins = {'repo_id': repo, 'revision': info.sha, 'weights_sha256': weights.lfs.sha256}
    if old and pins != old: raise ValueError('Los metadatos oficiales difieren del fijado anterior')
    atomic(pin_path, canonical(pins))
    checkpoint = Path('/content/models') / kind
    subprocess.run([sys.executable, 'tools/research/phase4/download_models.py', '--model', kind,
                    '--revision', info.sha, '--out', str(checkpoint)], check=True)
    hashes = {n: sha_file(checkpoint / n) for n in ('config.json', 'model.safetensors', 'preprocessor_config.json')}
    if hashes['model.safetensors'] != pins['weights_sha256']:
        raise ValueError('Los pesos descargados no coinciden con el digest oficial')
    return checkpoint, info.sha, hashes

def run_model(kind):
    if kind == 'openclip':
        import urllib.request
        checkpoint = Path('/content/vit_b_32-quickgelu-laion400m_e31-d867053b.pt')
        if not checkpoint.exists():
            urllib.request.urlretrieve('https://github.com/mlfoundations/open_clip/releases/download/'
                                       'v0.2-weights/vit_b_32-quickgelu-laion400m_e31-d867053b.pt', checkpoint)
        revision = 'v0.2-weights'
        adapter = load_frozen(kind, checkpoint, device=device)
        pins = {'revision': revision, 'checkpoint_sha256': adapter.provenance['weights_sha256']}
    else:
        checkpoint, revision, hashes = official_checkpoint(kind)
        adapter = load_frozen(kind, checkpoint, config_sha256=hashes['config.json'],
                              weights_sha256=hashes['model.safetensors'],
                              processor_sha256=hashes['preprocessor_config.json'],
                              device=device, model_revision=revision)
        pins = {'revision': revision, 'checkpoint_sha256': hashes['model.safetensors'],
                'processor_sha256': hashes['preprocessor_config.json'], 'config_sha256': hashes['config.json']}
    state = run_extraction(JOB_PATH, Path(WORK_DIR) / kind, adapter, model_revision=revision,
                           device=device, batch_size=BATCH_SIZE)
    bundle = Path(OUT_DIR) / f'{kind}_development.zip'
    index = export_bundle(JOB_PATH, Path(WORK_DIR) / kind, bundle)
    pins.update(model=kind, bundle=bundle.name, bundle_sha256=sha_file(bundle), status=state['status'],
                device=device, torch=torch.__version__, objects=len(index['objects']))
    return pins

RESULTS = {}
for kind in MODELS:
    print('==', kind)
    RESULTS[kind] = run_model(kind)
    print(json.dumps(RESULTS[kind], indent=1))
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
(Path(OUT_DIR) / 'pins.json').write_text(json.dumps(RESULTS, indent=1, sort_keys=True) + '\n')
for kind, pin in RESULTS.items():
    flags = '' if kind == 'openclip' else (f" --expected-revision {pin['revision']}"
             f" --expected-checkpoint-sha256 {pin['checkpoint_sha256']}"
             f" --expected-processor-sha256 {pin['processor_sha256']}")
    print(f"\nImportar {kind} en el repositorio:\nPYTHONPATH=. python tools/research/phase4/import_embedding_bundle.py "
          f"--job research/phase4/embedding_experiment/development-images.zip --bundle {pin['bundle']} "
          f"--destination imported-{kind} --analysis-out {kind}_analysis.json{flags}")


In [ ]:
# Si se ejecutó DINOv2, compara sus vectores globales con la referencia CPU del repositorio.
import io, zipfile
import numpy as np
if 'dinov2' in RESULTS:
    reference = np.load('research/phase4/embedding_experiment/dinov2/global_embeddings.npz', allow_pickle=False)
    with zipfile.ZipFile(Path(OUT_DIR) / RESULTS['dinov2']['bundle']) as bundle:
        index = json.loads(bundle.read('index.json'))
        ran = {row['object_id']: np.load(io.BytesIO(bundle.read(row['file'])))['global_embeddings'] for row in index['objects']}
    worst = 1.0
    for name, expected in zip(reference['ids'].tolist(), reference['global_embeddings']):
        got = ran[name]
        cosine = (got * expected).sum(axis=1) / np.linalg.norm(got, axis=1) / np.linalg.norm(expected, axis=1)
        worst = min(worst, float(cosine.min()))
    print(f'Coseno mínimo frente a la referencia CPU: {worst:.6f} (1 en CPU; en GPU cerca de 1 por redondeos)')
    if worst < 0.999: print('AVISO: diferencia mayor de lo esperado; revisa versión de torch/transformers y procesado.')


In [ ]:
from google.colab import files
for pin in RESULTS.values():
    files.download(str(Path(OUT_DIR) / pin['bundle']))
files.download(str(Path(OUT_DIR) / 'pins.json'))


## Llevar los resultados al repositorio

La celda de ejecución imprime, por modelo, la orden de importación con la revisión y los hashes que la
verifican. El importador comprueba la seguridad del ZIP, la identidad del trabajo, el orden de las vistas y
los hashes de cada fichero antes de lanzar el análisis de desarrollo, y deja el análisis comparable con
`research/phase4/embedding_experiment/foundation_comparison.json`:

```bash
PYTHONPATH=. python -m research_grasping.phase4.model_comparison \
  --inputs research/phase4/embedding_experiment/openclip_analysis.json \
           research/phase4/embedding_experiment/dinov2/analysis.json dinov3_analysis.json \
  --out foundation_comparison.json
```

Los hashes prueban integridad, no que un entorno ajeno usara de verdad el modelo declarado: audita la
procedencia antes de usar los resultados científicamente.
